# Cluster Graph Visualisation

Builds and analyses three complementary Leiden-based cluster graphs from the chess research corpus:

- **Keyword graph** — articles linked by shared keywords; clusters reveal thematic research communities.
- **Author graph** — articles linked by shared authors; clusters reveal research generations and collaboration circles.
- **Combined graph** — hybrid of keyword and authorship edges for a unified view.

Each clustering is visualised as an interactive Pyvis HTML network and analysed for temporal coherence, inter-cluster connectivity, keyword prominence trends, and foundational anchor papers.

**Output:** GraphML files and HTML visualisations in 

In [ ]:
# Leiden clustering based on the same data pipeline as graph_vis.ipynb
import ast
import itertools
import colorsys
from collections import defaultdict

import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
from pyvis.network import Network
from pathlib import Path
import math
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.patches as mpatches
import csv
import matplotlib.cm as cm
import pandas as pd
import plotly.graph_objects as go
from collections import Counter

# Input data (same as graph_vis.ipynb)
# Directory roots
OUTPUT_DIR = Path("..").resolve() / "output"
KW_DIR = Path("..").resolve() / "keywords_results"

META_PATH = OUTPUT_DIR / "metadata_enriched_after_openalex_year.csv"
KW_PATH = KW_DIR / "keywords_normalized_deduplicated_no_stopwords.csv"

# Output files
GRAPHML_OUT = OUTPUT_DIR / "articles_leiden_clusters.graphml"
HTML_OUT = OUTPUT_DIR / "articles_leiden_clusters.html"

# Build/pruning params (same defaults)
MIN_SHARED_KEYWORDS = 3
MIN_EDGE_STRENGTH = 3.2
MIN_MAX_PAIR_SCORE = 3.5
TOP_K_PER_NODE = 8
USE_MUTUAL_KNN = True
AUTHOR_BOOST = 0.25
INCLUDE_AUTHOR_ONLY_EDGES = False
MAX_TOTAL_EDGES = None
LEIDEN_RESOLUTION = 1.0
LEIDEN_SEED = 42


def normalize_id_from_pdf(pdf_name: str) -> str:
    """Normalize article ID from a PDF filename."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    if s.lower().endswith(".pdf"):
        s = s[:-4]
    return s.lower()


def normalize_id_from_filename(filename: str) -> str:
    """Normalize article ID from a plain filename."""
    if pd.isna(filename):
        return ""
    return str(filename).strip().lower()


def split_authors(raw_authors: str):
    """Split a semicolon-delimited author string into a list of stripped names."""
    if pd.isna(raw_authors):
        return []
    parts = [a.strip() for a in str(raw_authors).split(";")]
    return [p for p in parts if p]


def pick_title(row):
    """Return the best available title for an article row."""
    t = row.get("crossref_title")
    if isinstance(t, str) and t.strip():
        return t.strip()
    t = row.get("original_title")
    if isinstance(t, str) and t.strip():
        return t.strip()
    return row.get("pdf_file", "unknown_title")


def pick_authors(row):
    """Return the best available author list for an article row."""
    a = row.get("crossref_authors")
    if isinstance(a, str) and a.strip():
        return split_authors(a)
    return split_authors(row.get("original_authors"))


def parse_keywords_with_strength(cell):
    """Parse keyword-strength pairs from a raw cell string."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
    except Exception:
        return []

    out = []
    for item in parsed:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            kw = str(item[0]).strip().lower()
            try:
                strength = float(item[1])
            except Exception:
                continue
            if kw:
                out.append((kw, strength))
    return out


# Load and merge data
meta = pd.read_csv(META_PATH)
kw = pd.read_csv(KW_PATH)

meta["article_id"] = meta["pdf_file"].map(normalize_id_from_pdf)
kw["article_id"] = kw["filename"].map(normalize_id_from_filename)

meta_small = meta[["article_id", "pdf_file", "original_title", "crossref_title", "original_authors", "crossref_authors"]].copy()
kw["keyword_items"] = kw["keywords_without_stopwords"].map(parse_keywords_with_strength)

articles = meta_small.merge(
    kw[["article_id", "keyword_items"]],
    on="article_id",
    how="left"
)
articles["keyword_items"] = articles["keyword_items"].apply(lambda x: x if isinstance(x, list) else [])
articles["title"] = articles.apply(pick_title, axis=1)
articles["authors"] = articles.apply(pick_authors, axis=1)

print("articles:", len(articles))


# Build graph
G = nx.Graph()
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    title = row["title"]
    authors = row["authors"]
    kw_items = row["keyword_items"]

    G.add_node(
        article_id,
        label=title,
        title=title,
        pdf_file=str(row["pdf_file"]),
        authors="|".join(authors),
        n_authors=len(authors),
        n_keywords=len(kw_items),
    )

article_kw_strength = {}
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    tmp = defaultdict(list)
    for kw_term, kw_strength in row["keyword_items"]:
        tmp[kw_term].append(float(kw_strength))
    article_kw_strength[article_id] = {k: sum(v) / len(v) for k, v in tmp.items()}

author_to_articles = defaultdict(set)
keyword_to_articles = defaultdict(list)

for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])

    for author in row["authors"]:
        author_to_articles[author.lower()].add(article_id)

    for kw_term, kw_strength in article_kw_strength[article_id].items():
        keyword_to_articles[kw_term].append((article_id, kw_strength))

pair_acc = {}


def pair_key(a, b):
    """Return a canonical (min, max) pair key for two node IDs."""
    return (a, b) if a < b else (b, a)


for kw_term, art_strength_list in keyword_to_articles.items():
    if len(art_strength_list) < 2:
        continue

    for (u, su), (v, sv) in itertools.combinations(art_strength_list, 2):
        k = pair_key(u, v)
        if k not in pair_acc:
            pair_acc[k] = {
                "kw_scores": [],
                "kw_terms": set(),
                "shared_authors": set(),
            }
        pair_score = (float(su) + float(sv)) / 2.0
        pair_acc[k]["kw_scores"].append(pair_score)
        pair_acc[k]["kw_terms"].add(kw_term)

for author, arts in author_to_articles.items():
    arts = sorted(arts)
    if len(arts) < 2:
        continue
    for u, v in itertools.combinations(arts, 2):
        k = pair_key(u, v)
        if k not in pair_acc and not INCLUDE_AUTHOR_ONLY_EDGES:
            continue
        if k not in pair_acc:
            pair_acc[k] = {
                "kw_scores": [],
                "kw_terms": set(),
                "shared_authors": set(),
            }
        pair_acc[k]["shared_authors"].add(author)

candidate_edges = []
for (u, v), acc in pair_acc.items():
    kw_scores = acc["kw_scores"]
    kw_terms = sorted(acc["kw_terms"])
    kw_count = len(kw_terms)

    shared_authors = sorted(acc["shared_authors"])
    shared_authors_count = len(shared_authors)

    has_kw = kw_count > 0
    has_author = shared_authors_count > 0

    if has_kw:
        keyword_strength_mean = sum(kw_scores) / len(kw_scores)
        max_pair_score = max(kw_scores)
    else:
        keyword_strength_mean = -1.0
        max_pair_score = -1.0

    final_score = keyword_strength_mean
    if has_kw and has_author:
        final_score += AUTHOR_BOOST * min(shared_authors_count, 2)

    keep = False
    if has_kw:
        keep = (
            (kw_count >= MIN_SHARED_KEYWORDS)
            and (keyword_strength_mean >= MIN_EDGE_STRENGTH)
            and (max_pair_score >= MIN_MAX_PAIR_SCORE)
        )
    elif has_author and INCLUDE_AUTHOR_ONLY_EDGES:
        keep = True

    if not keep:
        continue

    edge_type = "author+keyword" if (has_kw and has_author) else ("keyword" if has_kw else "author")

    candidate_edges.append({
        "u": u,
        "v": v,
        "edge_type": edge_type,
        "strength": float(final_score),
        "keyword_strength_mean": float(keyword_strength_mean),
        "shared_keywords_count": kw_count,
        "shared_keywords": "|".join(kw_terms),
        "shared_authors_count": shared_authors_count,
        "shared_authors": "|".join(shared_authors),
    })

adj = defaultdict(list)
for idx, e in enumerate(candidate_edges):
    adj[e["u"]].append((idx, e["strength"]))
    adj[e["v"]].append((idx, e["strength"]))

top_ids_by_node = {}
for node, arr in adj.items():
    arr_sorted = sorted(arr, key=lambda x: x[1], reverse=True)
    top_ids_by_node[node] = {idx for idx, _ in arr_sorted[:TOP_K_PER_NODE]}

kept_edge_ids = set()
for idx, e in enumerate(candidate_edges):
    u, v = e["u"], e["v"]
    in_u = idx in top_ids_by_node.get(u, set())
    in_v = idx in top_ids_by_node.get(v, set())

    if USE_MUTUAL_KNN:
        if in_u and in_v:
            kept_edge_ids.add(idx)
    else:
        if in_u or in_v:
            kept_edge_ids.add(idx)

kept_edges = [candidate_edges[idx] for idx in kept_edge_ids]
kept_edges = sorted(kept_edges, key=lambda x: x["strength"], reverse=True)
if MAX_TOTAL_EDGES is not None:
    kept_edges = kept_edges[:MAX_TOTAL_EDGES]

for e in kept_edges:
    G.add_edge(
        e["u"],
        e["v"],
        edge_type=e["edge_type"],
        strength=e["strength"],
        keyword_strength_mean=e["keyword_strength_mean"],
        shared_keywords_count=e["shared_keywords_count"],
        shared_keywords=e["shared_keywords"],
        shared_authors_count=e["shared_authors_count"],
        shared_authors=e["shared_authors"],
    )

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())


# Leiden clustering on connected portion (isolates are marked separately)
connected_nodes = [n for n, deg in G.degree() if deg > 0]
isolated_nodes = [n for n, deg in G.degree() if deg == 0]
G_connected = G.subgraph(connected_nodes).copy()

_ig_G = ig.Graph.from_networkx(G_connected)
_node_names = _ig_G.vs["_nx_name"]
_partition = leidenalg.find_partition(
    _ig_G,
    leidenalg.RBConfigurationVertexPartition,
    weights="strength",
    resolution_parameter=LEIDEN_RESOLUTION,
    seed=LEIDEN_SEED,
)
communities = [frozenset(_node_names[v] for v in m) for m in _partition]

node_to_cluster = {}
for cluster_id, nodes in enumerate(communities):
    for node in nodes:
        node_to_cluster[node] = cluster_id

# Keep isolated nodes out of topical clusters
for node in isolated_nodes:
    node_to_cluster[node] = -1

nx.set_node_attributes(G, node_to_cluster, "cluster")

cluster_sizes = sorted((len(c) for c in communities), reverse=True)
print("connected nodes:", len(connected_nodes))
print("isolated nodes:", len(isolated_nodes))
print("leiden communities (connected graph):", len(communities))
print("largest community size:", cluster_sizes[0] if cluster_sizes else 0)


# Export clustered graph
nx.write_graphml(G, GRAPHML_OUT)
print("GraphML written to:", GRAPHML_OUT)


# Cluster report: size + top keywords + representative titles
CLUSTER_REPORT_OUT = OUTPUT_DIR / "articles_leiden_cluster_report.csv"

AUTH_GRAPHML_OUT = OUTPUT_DIR / "articles_leiden_authors.graphml"
AUTH_HTML_OUT = OUTPUT_DIR / "articles_leiden_authors.html"
AUTH_REPORT_OUT = OUTPUT_DIR / "articles_leiden_authors_report.csv"
COMB_GRAPHML_OUT = OUTPUT_DIR / "articles_leiden_combined.graphml"
COMB_HTML_OUT = OUTPUT_DIR / "articles_leiden_combined.html"
COMB_REPORT_OUT = OUTPUT_DIR / "articles_leiden_combined_report.csv"
COMPARISON_OUT = OUTPUT_DIR / "articles_leiden_views_comparison.csv"
YEAR_DEV_OUT = OUTPUT_DIR / "articles_leiden_cluster_year_deviation.csv"
TOP10_YEAR_COHERENCE_OUT = OUTPUT_DIR / "articles_leiden_top10_time_clusters.csv"
KW_DISTINCTIVE_OUT = OUTPUT_DIR / "keyword_cluster_distinctive_terms.csv"
KW_REPRESENTATIVE_OUT = OUTPUT_DIR / "keyword_cluster_representative_papers.csv"
KW_BRIDGES_OUT = OUTPUT_DIR / "keyword_cluster_bridge_articles.csv"
KW_TIME_OUT = OUTPUT_DIR / "keyword_cluster_time_trends.csv"
KW_CLUSTER_GRAPHML_OUT = OUTPUT_DIR / "keyword_cluster_intercluster.graphml"
KW_CLUSTER_HTML_OUT = OUTPUT_DIR / "keyword_cluster_intercluster.html"
YEAR_PERIOD_SUMMARY_OUT = OUTPUT_DIR / "articles_year_period_summary.csv"
YEAR_PERIOD_DIST_OUT = OUTPUT_DIR / "articles_year_period_distribution.csv"
YEAR_PERIOD_PLOT_OUT = OUTPUT_DIR / "articles_year_period_distribution.png"
OUTPUT_PATH = OUTPUT_DIR / "keyword_temporal_heatmap.png"
ERA_HTML_OUT = OUTPUT_DIR / "articles_leiden_era_colored.html"
KW_EMG_PATH = OUTPUT_DIR / "graph/keyword_emergence.csv"
ERA_FLOW_PATH = OUTPUT_DIR / "graph/cross_era_flow.csv"
KW_CLASS_OUT = OUTPUT_DIR / "rq4_keyword_classification.csv"
YEAR_HTML_OUT = OUTPUT_DIR / "articles_leiden_year_colored.html"
CLUSTER_REPORT_PATH = OUTPUT_DIR / "articles_leiden_cluster_report.csv"
CR_PATH = OUTPUT_DIR / "articles_leiden_cluster_report.csv"
ARC_OUT = OUTPUT_DIR / "author_career_arcs.csv"
SPEC_OUT = OUTPUT_DIR / "keyword_specificity_spectrum.csv"
SANKEY_OUT = OUTPUT_DIR / "sankey_kw_auth_comb.html"
SANKEY_PNG = OUTPUT_DIR / "sankey_kw_auth_comb.png"
VENUE_OUT = OUTPUT_DIR / "venue_concentration_per_cluster.csv"

cluster_to_nodes = {cid: sorted(list(nodes)) for cid, nodes in enumerate(communities)}

node_title = {}
node_keywords = {}
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    node_title[article_id] = row["title"]
    node_keywords[article_id] = article_kw_strength.get(article_id, {})

report_rows = []
cluster_topic_label = {-1: "isolated/no-links"}
for cluster_id, nodes in cluster_to_nodes.items():
    kw_agg = defaultdict(float)
    for n in nodes:
        for kw_term, kw_strength in node_keywords.get(n, {}).items():
            kw_agg[kw_term] += float(kw_strength)

    top_keywords = sorted(kw_agg.items(), key=lambda x: x[1], reverse=True)[:12]
    top_keywords_str = " | ".join([f"{k}:{v:.2f}" for k, v in top_keywords])

    # Human-readable cluster topic, e.g. "ml, neural network, classification"
    topic_terms = [k for k, _ in top_keywords[:3]]
    if topic_terms:
        topic_label = ", ".join(topic_terms)
    else:
        topic_label = "no keywords"

    reps = [node_title.get(n, n) for n in nodes[:5]]
    reps_str = " | ".join(reps)

    cluster_topic_label[cluster_id] = topic_label
    report_rows.append({
        "cluster_id": cluster_id,
        "cluster_size": len(nodes),
        "topic_label": topic_label,
        "top_keywords": top_keywords_str,
        "representative_titles": reps_str,
    })

cluster_report = pd.DataFrame(report_rows).sort_values("cluster_size", ascending=False)

# Add one explicit line for isolated nodes
if isolated_nodes:
    isolated_row = {
        "cluster_id": -1,
        "cluster_size": len(isolated_nodes),
        "topic_label": "isolated/no-links",
        "top_keywords": "",
        "representative_titles": " | ".join([node_title.get(n, n) for n in isolated_nodes[:5]]),
    }
    cluster_report = pd.concat([cluster_report, pd.DataFrame([isolated_row])], ignore_index=True)

cluster_report.to_csv(CLUSTER_REPORT_OUT, index=False)

# Save topic label into node attributes for tooltip/analysis
for n in G.nodes():
    cid = int(G.nodes[n].get("cluster", -1))
    G.nodes[n]["cluster_topic"] = cluster_topic_label.get(cid, "unknown")

print("Cluster report written to:", CLUSTER_REPORT_OUT)
print("Top clusters preview:")
print(cluster_report[["cluster_id", "cluster_size", "topic_label"]].head(20).to_string(index=False))


# Cluster-aware visualization (better separation and readability)
def color_for_cluster(cluster_id: int) -> str:
    """Return a deterministic hex color for a cluster ID using golden-ratio hue stepping."""
    # Generate deterministic near-unique colors using golden-ratio hue stepping
    if cluster_id < 0:
        return "#95a5a6"
    h = (cluster_id * 0.618033988749895) % 1.0
    s = 0.55 + 0.35 * ((cluster_id % 3) / 2.0)
    v = 0.75 + 0.20 * ((cluster_id % 2) / 1.0)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))

# Visualization graph: hide isolates to avoid clutter and improve cluster separation
G_vis = G.subgraph([n for n in G.nodes() if int(G.nodes[n].get("cluster", -1)) >= 0]).copy()

net = Network(height="950px", width="100%", bgcolor="white", font_color="black", notebook=False)

# Stronger repulsion and longer springs to separate cluster islands
net.barnes_hut(
    gravity=-42000,
    central_gravity=0.06,
    spring_length=260,
    spring_strength=0.012,
    damping=0.86,
    overlap=0.2,
)

for n, nd in G_vis.nodes(data=True):
    title = nd.get("title", n)
    cluster_id = int(nd.get("cluster", -1))
    topic_label = nd.get("cluster_topic", "unknown")
    color = color_for_cluster(cluster_id)
    node_size = 8 + min(24, G_vis.degree(n))

    net.add_node(
        n,
        label=f"C{cluster_id}",
        group=cluster_id,
        title=(
            f"{title}"
            f"<br>Cluster: C{cluster_id}"
            f"<br>Topic: {topic_label}"
            f"<br>Degree: {G_vis.degree(n)}"
            f"<br>Authors: {nd.get('authors', '')}"
        ),
        value=node_size,
        color=color,
    )

for u, v, ed in G_vis.edges(data=True):
    strength = float(ed.get("strength", 1.0))
    cu = int(G_vis.nodes[u].get("cluster", -1))
    cv = int(G_vis.nodes[v].get("cluster", -1))
    intra = cu == cv and cu >= 0

    if intra:
        edge_color = color_for_cluster(cu)
        edge_width = 1.3 + 0.45 * max(0.0, strength - 2.0)
        edge_opacity = 0.65
    else:
        edge_color = "#666666"
        edge_width = 0.4
        edge_opacity = 0.12

    net.add_edge(
        u,
        v,
        width=edge_width,
        color={"color": edge_color, "opacity": edge_opacity},
        title=f"score={strength:.3f} | intra_cluster={intra}",
    )

# Slightly stronger repulsion helps cluster islands separate visually
net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}
  },
  "nodes": {
    "font": {"size": 12}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 150
  }
}
""")

net.write_html(HTML_OUT, open_browser=False)
print("HTML written to:", HTML_OUT)
print("Visualized nodes (non-isolated only):", G_vis.number_of_nodes())
print("Visualized edges:", G_vis.number_of_edges())

articles: 2376
nodes: 2376
edges: 1819
connected nodes: 1177
isolated nodes: 1199
leiden communities (connected graph): 125
largest community size: 54
GraphML written to: /home/john/repos/leno4ka/output/articles_leiden_clusters.graphml
Cluster report written to: /home/john/repos/leno4ka/output/articles_leiden_cluster_report.csv
Top clusters preview:
 cluster_id  cluster_size                                                                            topic_label
          0            54                                                 work stealing, load balancing, speedup
          1            48                                   mixed strategy, backward induction, winning position
          2            48                                              policy head, value head, residual network
          3            47                                       turing machine, problem space, production system
          4            46                                 discrimination net, exper

**Author co-authorship graph.** Runs the same Leiden algorithm on a co-authorship graph, grouping researchers who frequently publish together into author communities.

In [ ]:
# Author-only Leiden clustering (separate visualization)
# Reuses `articles`, helper functions, and visualization utilities from Cell 0.

MIN_SHARED_AUTHORS = 1
AUTHOR_EDGE_WEIGHT_PER_SHARED_AUTHOR = 1.0
AUTHOR_LEIDEN_RESOLUTION = 1.0
AUTHOR_LEIDEN_SEED = 42


# Build author graph
G_author = nx.Graph()
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    G_author.add_node(
        article_id,
        title=row["title"],
        authors="|".join(row["authors"]),
        n_authors=len(row["authors"]),
    )

author_to_articles = defaultdict(set)
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    for a in row["authors"]:
        author_to_articles[str(a).strip().lower()].add(article_id)

pair_shared_authors = defaultdict(set)
for author, art_ids in author_to_articles.items():
    art_ids = sorted(art_ids)
    if len(art_ids) < 2:
        continue
    for i in range(len(art_ids)):
        for j in range(i + 1, len(art_ids)):
            u, v = art_ids[i], art_ids[j]
            k = (u, v) if u < v else (v, u)
            pair_shared_authors[k].add(author)

for (u, v), shared in pair_shared_authors.items():
    shared_count = len(shared)
    if shared_count < MIN_SHARED_AUTHORS:
        continue
    strength = AUTHOR_EDGE_WEIGHT_PER_SHARED_AUTHOR * shared_count
    G_author.add_edge(
        u,
        v,
        strength=float(strength),
        shared_authors_count=int(shared_count),
        shared_authors="|".join(sorted(shared)),
        edge_type="author",
    )

print("[authors] nodes:", G_author.number_of_nodes())
print("[authors] edges:", G_author.number_of_edges())


# Leiden on connected author graph
connected_nodes = [n for n, d in G_author.degree() if d > 0]
isolated_nodes = [n for n, d in G_author.degree() if d == 0]
G_author_connected = G_author.subgraph(connected_nodes).copy()

_ig_G_auth = ig.Graph.from_networkx(G_author_connected)
_node_names_auth = _ig_G_auth.vs["_nx_name"]
_partition_auth = leidenalg.find_partition(
    _ig_G_auth,
    leidenalg.RBConfigurationVertexPartition,
    weights="strength",
    resolution_parameter=AUTHOR_LEIDEN_RESOLUTION,
    seed=AUTHOR_LEIDEN_SEED,
)
communities = [frozenset(_node_names_auth[v] for v in m) for m in _partition_auth]

node_to_cluster = {}
for cid, nodes in enumerate(communities):
    for n in nodes:
        node_to_cluster[n] = cid
for n in isolated_nodes:
    node_to_cluster[n] = -1

nx.set_node_attributes(G_author, node_to_cluster, "cluster")


# Cluster report (author topics are representative authors)
cluster_to_nodes = {cid: sorted(list(nodes)) for cid, nodes in enumerate(communities)}
report_rows = []
cluster_topic_label = {-1: "isolated/no-links"}

for cid, nodes in cluster_to_nodes.items():
    author_freq = defaultdict(int)
    for n in nodes:
        for a in str(G_author.nodes[n].get("authors", "")).split("|"):
            a = a.strip()
            if a:
                author_freq[a] += 1

    top_authors = sorted(author_freq.items(), key=lambda x: x[1], reverse=True)[:5]
    topic_label = ", ".join([a for a, _ in top_authors[:3]]) if top_authors else "no authors"
    cluster_topic_label[cid] = topic_label

    rep_titles = [G_author.nodes[n].get("title", n) for n in nodes[:5]]
    report_rows.append(
        {
            "cluster_id": cid,
            "cluster_size": len(nodes),
            "topic_label": topic_label,
            "top_authors": " | ".join([f"{a}:{c}" for a, c in top_authors]),
            "representative_titles": " | ".join(rep_titles),
        }
    )

report = pd.DataFrame(report_rows).sort_values("cluster_size", ascending=False)
if isolated_nodes:
    report = pd.concat(
        [
            report,
            pd.DataFrame(
                [
                    {
                        "cluster_id": -1,
                        "cluster_size": len(isolated_nodes),
                        "topic_label": "isolated/no-links",
                        "top_authors": "",
                        "representative_titles": " | ".join(
                            [G_author.nodes[n].get("title", n) for n in isolated_nodes[:5]]
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

report.to_csv(AUTH_REPORT_OUT, index=False)
print("[authors] report:", AUTH_REPORT_OUT)
print(report[["cluster_id", "cluster_size", "topic_label"]].head(15).to_string(index=False))


# Save cluster topic on nodes
for n in G_author.nodes():
    cid = int(G_author.nodes[n].get("cluster", -1))
    G_author.nodes[n]["cluster_topic"] = cluster_topic_label.get(cid, "unknown")

nx.write_graphml(G_author, AUTH_GRAPHML_OUT)
print("[authors] graphml:", AUTH_GRAPHML_OUT)


# Visualization (hide isolates)
G_vis = G_author.subgraph([n for n in G_author.nodes() if int(G_author.nodes[n].get("cluster", -1)) >= 0]).copy()
net = Network(height="950px", width="100%", bgcolor="white", font_color="black", notebook=False)
net.barnes_hut(
    gravity=-42000,
    central_gravity=0.06,
    spring_length=260,
    spring_strength=0.012,
    damping=0.86,
    overlap=0.2,
)

for n, nd in G_vis.nodes(data=True):
    cid = int(nd.get("cluster", -1))
    color = color_for_cluster(cid)
    net.add_node(
        n,
        label=f"C{cid}",
        group=cid,
        title=(
            f"{nd.get('title', n)}"
            f"<br>Cluster: C{cid}"
            f"<br>Topic: {nd.get('cluster_topic', 'unknown')}"
            f"<br>Authors: {nd.get('authors', '')}"
        ),
        value=8 + min(24, G_vis.degree(n)),
        color=color,
    )

for u, v, ed in G_vis.edges(data=True):
    cu = int(G_vis.nodes[u].get("cluster", -1))
    cv = int(G_vis.nodes[v].get("cluster", -1))
    intra = cu == cv and cu >= 0
    strength = float(ed.get("strength", 1.0))
    if intra:
        edge_color = color_for_cluster(cu)
        edge_width = 1.2 + 0.5 * min(3.0, strength)
        edge_opacity = 0.65
    else:
        edge_color = "#666666"
        edge_width = 0.4
        edge_opacity = 0.12
    net.add_edge(
        u,
        v,
        width=edge_width,
        color={"color": edge_color, "opacity": edge_opacity},
        title=f"shared_authors={int(ed.get('shared_authors_count', 0))}",
    )

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}
  },
  "interaction": {"hover": true, "tooltipDelay": 120}
}
""")

net.write_html(AUTH_HTML_OUT, open_browser=False)
print("[authors] html:", AUTH_HTML_OUT)
print("[authors] visualized nodes:", G_vis.number_of_nodes())
print("[authors] visualized edges:", G_vis.number_of_edges())

[authors] nodes: 2376
[authors] edges: 21820
[authors] report: /home/john/repos/leno4ka/output/articles_leiden_authors_report.csv
 cluster_id  cluster_size                                           topic_label
          0           178                                               Nan Nan
          1            88    Fernand Gobet, Guillermo Campitelli, Merim Bilalić
          2            62            Jonathan Schaeffer, T Marsland, Aske Plaat
          3            59                  Ivan Bratko, Matej Guid, Azlan Iqbal
          4            41     Hiroyuki Iida, Nathan S. Netanyahu, Makoto Sakuta
          5            40         Hans Berliner, Murray Campbell, T.A. Marsland
          6            31               Guy Haworth, Joe Hurd, Kenneth W. Regan
          7            24               David Silver, Matthew Lai, Marc Lanctot
          8            21   Neil Charness, K. Anders Ericsson, Eyal M. Reingold
          9            21 Paolo Ciancarini, Gian Piero Favini, Andrea 

**Combined keyword + author graph.** Merges keyword similarity and co-authorship edges with a configurable weight blend, then clusters the resulting hybrid graph with Leiden.

In [ ]:
# Combined keyword+author Leiden clustering (separate visualization)
# Reuses `articles`, `article_kw_strength`, and helper functions from Cell 0.

# Keep combination simple and interpretable
ALPHA_KEYWORD = 0.8
BETA_AUTHOR = 0.2
MIN_SHARED_KEYWORDS = 1
MIN_KEYWORD_STRENGTH = 2.6
TOP_K_PER_NODE = 14
USE_MUTUAL_KNN = False
COMBINED_LEIDEN_RESOLUTION = 1.0
COMBINED_LEIDEN_SEED = 42


# Build combined graph
G_comb = nx.Graph()
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    G_comb.add_node(
        article_id,
        title=row["title"],
        authors="|".join(row["authors"]),
        n_authors=len(row["authors"]),
        n_keywords=len(row["keyword_items"]),
    )

# Keyword-based pair accumulation
keyword_to_articles = defaultdict(list)
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    for kw_term, kw_strength in article_kw_strength.get(article_id, {}).items():
        keyword_to_articles[kw_term].append((article_id, float(kw_strength)))

pair_kw = defaultdict(list)
for kw_term, arr in keyword_to_articles.items():
    if len(arr) < 2:
        continue
    for (u, su), (v, sv) in itertools.combinations(arr, 2):
        k = (u, v) if u < v else (v, u)
        pair_kw[k].append((su + sv) / 2.0)

# Author overlap per pair
author_to_articles = defaultdict(set)
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    for a in row["authors"]:
        author_to_articles[str(a).strip().lower()].add(article_id)

pair_author_cnt = defaultdict(int)
for a, arts in author_to_articles.items():
    arts = sorted(arts)
    if len(arts) < 2:
        continue
    for i in range(len(arts)):
        for j in range(i + 1, len(arts)):
            u, v = arts[i], arts[j]
            k = (u, v) if u < v else (v, u)
            pair_author_cnt[k] += 1

# Candidate combined edges
candidate_edges = []
for pair, kw_scores in pair_kw.items():
    u, v = pair
    kw_count = len(kw_scores)
    kw_mean = sum(kw_scores) / kw_count if kw_count else 0.0
    if kw_count < MIN_SHARED_KEYWORDS or kw_mean < MIN_KEYWORD_STRENGTH:
        continue

    author_shared = int(pair_author_cnt.get(pair, 0))
    combined_strength = ALPHA_KEYWORD * kw_mean + BETA_AUTHOR * author_shared

    candidate_edges.append(
        {
            "u": u,
            "v": v,
            "strength": float(combined_strength),
            "keyword_strength_mean": float(kw_mean),
            "shared_keywords_count": int(kw_count),
            "shared_authors_count": int(author_shared),
        }
    )

# Top-k pruning
adj = defaultdict(list)
for idx, e in enumerate(candidate_edges):
    adj[e["u"]].append((idx, e["strength"]))
    adj[e["v"]].append((idx, e["strength"]))

keep_ids_by_node = {}
for n, arr in adj.items():
    arr_sorted = sorted(arr, key=lambda x: x[1], reverse=True)
    keep_ids_by_node[n] = {idx for idx, _ in arr_sorted[:TOP_K_PER_NODE]}

kept_ids = set()
for idx, e in enumerate(candidate_edges):
    in_u = idx in keep_ids_by_node.get(e["u"], set())
    in_v = idx in keep_ids_by_node.get(e["v"], set())
    if USE_MUTUAL_KNN:
        if in_u and in_v:
            kept_ids.add(idx)
    else:
        if in_u or in_v:
            kept_ids.add(idx)

for idx in kept_ids:
    e = candidate_edges[idx]
    G_comb.add_edge(
        e["u"],
        e["v"],
        strength=e["strength"],
        keyword_strength_mean=e["keyword_strength_mean"],
        shared_keywords_count=e["shared_keywords_count"],
        shared_authors_count=e["shared_authors_count"],
        edge_type="combined",
    )

print("[combined] nodes:", G_comb.number_of_nodes())
print("[combined] edges:", G_comb.number_of_edges())


# Leiden on connected combined graph
connected_nodes = [n for n, d in G_comb.degree() if d > 0]
isolated_nodes = [n for n, d in G_comb.degree() if d == 0]
G_comb_connected = G_comb.subgraph(connected_nodes).copy()

_ig_G_comb = ig.Graph.from_networkx(G_comb_connected)
_node_names_comb = _ig_G_comb.vs["_nx_name"]
_partition_comb = leidenalg.find_partition(
    _ig_G_comb,
    leidenalg.RBConfigurationVertexPartition,
    weights="strength",
    resolution_parameter=COMBINED_LEIDEN_RESOLUTION,
    seed=COMBINED_LEIDEN_SEED,
)
communities = [frozenset(_node_names_comb[v] for v in m) for m in _partition_comb]

node_to_cluster = {}
for cid, nodes in enumerate(communities):
    for n in nodes:
        node_to_cluster[n] = cid
for n in isolated_nodes:
    node_to_cluster[n] = -1

nx.set_node_attributes(G_comb, node_to_cluster, "cluster")


# Combined report (topic from keywords)
cluster_to_nodes = {cid: sorted(list(nodes)) for cid, nodes in enumerate(communities)}
report_rows = []
cluster_topic_label = {-1: "isolated/no-links"}

for cid, nodes in cluster_to_nodes.items():
    kw_agg = defaultdict(float)
    for n in nodes:
        for kw, s in article_kw_strength.get(n, {}).items():
            kw_agg[kw] += float(s)

    top_keywords = sorted(kw_agg.items(), key=lambda x: x[1], reverse=True)[:10]
    topic_label = ", ".join([k for k, _ in top_keywords[:3]]) if top_keywords else "no keywords"
    cluster_topic_label[cid] = topic_label

    report_rows.append(
        {
            "cluster_id": cid,
            "cluster_size": len(nodes),
            "topic_label": topic_label,
            "top_keywords": " | ".join([f"{k}:{v:.2f}" for k, v in top_keywords]),
            "representative_titles": " | ".join([G_comb.nodes[n].get("title", n) for n in nodes[:5]]),
        }
    )

report = pd.DataFrame(report_rows).sort_values("cluster_size", ascending=False)
if isolated_nodes:
    report = pd.concat(
        [
            report,
            pd.DataFrame(
                [
                    {
                        "cluster_id": -1,
                        "cluster_size": len(isolated_nodes),
                        "topic_label": "isolated/no-links",
                        "top_keywords": "",
                        "representative_titles": " | ".join([G_comb.nodes[n].get("title", n) for n in isolated_nodes[:5]]),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

report.to_csv(COMB_REPORT_OUT, index=False)
print("[combined] report:", COMB_REPORT_OUT)
print(report[["cluster_id", "cluster_size", "topic_label"]].head(15).to_string(index=False))


# Save topic on nodes and export
for n in G_comb.nodes():
    cid = int(G_comb.nodes[n].get("cluster", -1))
    G_comb.nodes[n]["cluster_topic"] = cluster_topic_label.get(cid, "unknown")

nx.write_graphml(G_comb, COMB_GRAPHML_OUT)
print("[combined] graphml:", COMB_GRAPHML_OUT)


# Visualization (hide isolates)
G_vis = G_comb.subgraph([n for n in G_comb.nodes() if int(G_comb.nodes[n].get("cluster", -1)) >= 0]).copy()
net = Network(height="950px", width="100%", bgcolor="white", font_color="black", notebook=False)
net.barnes_hut(
    gravity=-42000,
    central_gravity=0.06,
    spring_length=260,
    spring_strength=0.012,
    damping=0.86,
    overlap=0.2,
)

for n, nd in G_vis.nodes(data=True):
    cid = int(nd.get("cluster", -1))
    net.add_node(
        n,
        label=f"C{cid}",
        group=cid,
        title=(
            f"{nd.get('title', n)}"
            f"<br>Cluster: C{cid}"
            f"<br>Topic: {nd.get('cluster_topic', 'unknown')}"
            f"<br>n_keywords: {int(nd.get('n_keywords', 0))}"
            f"<br>n_authors: {int(nd.get('n_authors', 0))}"
        ),
        value=8 + min(24, G_vis.degree(n)),
        color=color_for_cluster(cid),
    )

for u, v, ed in G_vis.edges(data=True):
    cu = int(G_vis.nodes[u].get("cluster", -1))
    cv = int(G_vis.nodes[v].get("cluster", -1))
    intra = cu == cv and cu >= 0
    strength = float(ed.get("strength", 1.0))
    if intra:
        edge_color = color_for_cluster(cu)
        edge_width = 1.2 + 0.35 * max(0.0, strength - 2.0)
        edge_opacity = 0.65
    else:
        edge_color = "#666666"
        edge_width = 0.4
        edge_opacity = 0.12
    net.add_edge(
        u,
        v,
        width=edge_width,
        color={"color": edge_color, "opacity": edge_opacity},
        title=(
            f"combined={strength:.3f}"
            f" | kw_mean={float(ed.get('keyword_strength_mean', 0.0)):.3f}"
            f" | shared_authors={int(ed.get('shared_authors_count', 0))}"
        ),
    )

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}
  },
  "interaction": {"hover": true, "tooltipDelay": 120}
}
""")

net.write_html(COMB_HTML_OUT, open_browser=False)
print("[combined] html:", COMB_HTML_OUT)
print("[combined] visualized nodes:", G_vis.number_of_nodes())
print("[combined] visualized edges:", G_vis.number_of_edges())

[combined] nodes: 2376
[combined] edges: 22637
[combined] report: /home/john/repos/leno4ka/output/articles_leiden_combined_report.csv
 cluster_id  cluster_size                                                                   topic_label
          0           403                      critical thinking, cognitive development, search process
          1           246                                            computer vision, nnue, transformer
          2           239                  chess moves, aesthetic evaluation, chess problem composition
          3           230                          krk endgame, heuristic function, proof-number search
          4           220                                       speedup, work stealing, parallelization
          5           186                                 mtd(f), piece-square tables, fitness function
          6           183        counterfactual regret minimization, belief state, general game playing
          7           149         

**Cluster agreement analysis.** Compares how consistently each article is assigned across all three views (keyword, author, combined) using pairwise Jaccard similarity between cluster assignments.

In [ ]:
# Comparison of 3 clusterings: keywords vs authors vs combined

def _cluster_sets_from_graph(graph):
    """Build a dict mapping cluster_id to set of node IDs from graph node attributes."""
    clusters = {}
    for n, d in graph.nodes(data=True):
        cid = int(d.get("cluster", -1))
        clusters.setdefault(cid, set()).add(n)
    return clusters


def _safe_modularity(graph, clusters):
    """Compute modularity safely, returning NaN when the graph has no edges."""
    connected_clusters = [nodes for cid, nodes in clusters.items() if cid >= 0 and len(nodes) > 0]
    if graph.number_of_edges() == 0 or not connected_clusters:
        return float("nan")
    try:
        return float(nx.community.modularity(graph, connected_clusters, weight="strength"))
    except Exception:
        return float("nan")


def summarize_graph(name, graph):
    """Return a summary statistics dict for a clustered graph."""
    clusters = _cluster_sets_from_graph(graph)
    cluster_sizes = sorted([len(v) for k, v in clusters.items() if k >= 0], reverse=True)
    isolated = len(clusters.get(-1, set()))
    connected = graph.number_of_nodes() - isolated

    return {
        "view": name,
        "nodes": int(graph.number_of_nodes()),
        "edges": int(graph.number_of_edges()),
        "connected_nodes": int(connected),
        "isolated_nodes": int(isolated),
        "n_clusters": int(len([k for k in clusters.keys() if k >= 0])),
        "largest_cluster": int(cluster_sizes[0]) if cluster_sizes else 0,
        "mean_cluster_size": round(float(sum(cluster_sizes) / len(cluster_sizes)), 2) if cluster_sizes else 0.0,
        "modularity": round(_safe_modularity(graph, clusters), 4),
    }


comparison_rows = [
    summarize_graph("keywords", G),
    summarize_graph("authors", G_author),
    summarize_graph("combined", G_comb),
]

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(COMPARISON_OUT, index=False)
print("Comparison saved to:", COMPARISON_OUT)

    view  nodes  edges  connected_nodes  isolated_nodes  n_clusters  largest_cluster  mean_cluster_size  modularity
keywords   2376   1819             1177            1199         125               54               9.42         NaN
 authors   2376  21820             1360            1016         251              178               5.42         NaN
combined   2376  22637             2325              51          14              403             166.07         NaN
Comparison saved to: /home/john/repos/leno4ka/output/articles_leiden_views_comparison.csv


**Temporal cohesion per cluster.** Measures the spread of publication years within each cluster — tight clusters contain papers from a narrow time window, loose ones span decades.

In [ ]:
# Annual average deviation within cluster (publication year spread)

def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None
    # Keep only first 4-digit year if present
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        y = int(float(s))
        if 1500 <= y <= 2100:
            return y
    except Exception:
        return None
    return None


# Build article -> year map from metadata loaded in cell 0
article_year = {}
for _, row in meta.iterrows():
    article_id = normalize_id_from_pdf(row.get("pdf_file"))
    y_crossref = _to_year(row.get("crossref_year"))
    y_original = _to_year(row.get("original_year"))
    y_oa = _to_year(row.get("openalex_year"))
    article_year[article_id] = y_crossref if y_crossref is not None else (y_original if y_original is not None else y_oa)


def cluster_year_deviation_report(graph, view_name):
    """Build a DataFrame of year deviation statistics for each cluster in the graph."""
    rows = []
    cluster_ids = sorted({int(d.get("cluster", -1)) for _, d in graph.nodes(data=True)})

    for cid in cluster_ids:
        if cid < 0:
            continue
        nodes = [n for n, d in graph.nodes(data=True) if int(d.get("cluster", -1)) == cid]
        years = [article_year.get(n) for n in nodes]
        years = [y for y in years if y is not None]

        if len(years) == 0:
            mean_year = None
            median_year = None
            avg_abs_dev_mean = None
            avg_abs_dev_median = None
            std_year = None
            min_year = None
            max_year = None
        else:
            years_sorted = sorted(years)
            n_years = len(years_sorted)

            mean_year = float(sum(years_sorted) / n_years)

            if n_years % 2 == 1:
                median_year = float(years_sorted[n_years // 2])
            else:
                median_year = float((years_sorted[n_years // 2 - 1] + years_sorted[n_years // 2]) / 2.0)

            avg_abs_dev_mean = float(sum(abs(y - mean_year) for y in years_sorted) / n_years)
            avg_abs_dev_median = float(sum(abs(y - median_year) for y in years_sorted) / n_years)

            if n_years > 1:
                variance = sum((y - mean_year) ** 2 for y in years_sorted) / n_years
                std_year = float(variance ** 0.5)
            else:
                std_year = 0.0
            min_year = int(min(years_sorted))
            max_year = int(max(years_sorted))

        rows.append(
            {
                "view": view_name,
                "cluster_id": cid,
                "cluster_size": len(nodes),
                "articles_with_year": len(years),
                "mean_year": round(mean_year, 2) if mean_year is not None else None,
                "median_year": round(median_year, 2) if median_year is not None else None,
                "avg_abs_dev_from_mean": round(avg_abs_dev_mean, 3) if avg_abs_dev_mean is not None else None,
                "avg_abs_dev_from_median": round(avg_abs_dev_median, 3) if avg_abs_dev_median is not None else None,
                "std_year": round(std_year, 3) if std_year is not None else None,
                "min_year": min_year,
                "max_year": max_year,
            }
        )

    return pd.DataFrame(rows).sort_values(["cluster_size", "cluster_id"], ascending=[False, True])


year_kw = cluster_year_deviation_report(G, "keywords")
year_auth = cluster_year_deviation_report(G_author, "authors")
year_comb = cluster_year_deviation_report(G_comb, "combined")

year_report = pd.concat([year_kw, year_auth, year_comb], ignore_index=True)
year_report.to_csv(YEAR_DEV_OUT, index=False)

print("Annual deviation report saved to:", YEAR_DEV_OUT)
print("Top 15 largest clusters with year deviation:")
print(
    year_report[
        [
            "view",
            "cluster_id",
            "cluster_size",
            "articles_with_year",
            "mean_year",
            "median_year",
            "avg_abs_dev_from_mean",
            "avg_abs_dev_from_median",
            "std_year",
            "min_year",
            "max_year",
        ]
    ].head(15).to_string(index=False)
)

Annual deviation report saved to: /home/john/repos/leno4ka/output/articles_leiden_cluster_year_deviation.csv
Top 15 largest clusters with year deviation:
    view  cluster_id  cluster_size  articles_with_year  mean_year  median_year  avg_abs_dev_from_mean  avg_abs_dev_from_median  std_year  min_year  max_year
keywords           0            54                  45    1999.56       2000.0                 10.054                   10.044    13.854    1938.0    2020.0
keywords           1            48                  42    2003.26       2002.0                  9.417                    9.357    13.213    1966.0    2024.0
keywords           2            48                  43    2019.33       2022.0                  4.445                    4.023     6.231    1995.0    2026.0
keywords           3            47                  41    2003.27       2006.0                 10.528                   10.341    13.928    1957.0    2023.0
keywords           4            46                  40    200

**Top coherent and diverse clusters.** Ranks clusters by their temporal spread to surface the most time-focused research groups and the most temporally broad ones.

In [ ]:
# Top 10 most time-coherent and most time-diverse clusters per view

# Reuse year_report from previous cell if available; otherwise load from CSV
if "year_report" not in globals():
    year_report = pd.read_csv(OUTPUT_DIR / "articles_leiden_cluster_year_deviation.csv")

# Keep only clusters with at least 3 dated articles for more meaningful comparison
base = year_report.copy()
base = base[base["articles_with_year"].fillna(0) >= 3].copy()

outputs = []
for view_name in ["keywords", "authors", "combined"]:
    view_df = base[base["view"] == view_name].copy()
    if view_df.empty:
        continue

    most_coherent = view_df.sort_values(
        ["avg_abs_dev_from_median", "cluster_size"],
        ascending=[True, False],
    ).head(10).copy()
    most_coherent["ranking"] = "most_coherent"
    most_coherent["rank"] = range(1, len(most_coherent) + 1)

    most_diverse = view_df.sort_values(
        ["avg_abs_dev_from_median", "cluster_size"],
        ascending=[False, False],
    ).head(10).copy()
    most_diverse["ranking"] = "most_diverse"
    most_diverse["rank"] = range(1, len(most_diverse) + 1)

    outputs.extend([most_coherent, most_diverse])

    print(f"\n=== {view_name.upper()} | TOP 10 MOST TIME-COHERENT ===")
    print(
        most_coherent[
            [
                "rank",
                "cluster_id",
                "cluster_size",
                "articles_with_year",
                "median_year",
                "avg_abs_dev_from_median",
                "min_year",
                "max_year",
            ]
        ].to_string(index=False)
    )

    print(f"\n=== {view_name.upper()} | TOP 10 MOST TIME-DIVERSE ===")
    print(
        most_diverse[
            [
                "rank",
                "cluster_id",
                "cluster_size",
                "articles_with_year",
                "median_year",
                "avg_abs_dev_from_median",
                "min_year",
                "max_year",
            ]
        ].to_string(index=False)
    )


top10_df = pd.concat(outputs, ignore_index=True)
top10_df.to_csv(TOP10_YEAR_COHERENCE_OUT, index=False)
print("\nTop-10 time coherence/diversity table saved to:", TOP10_YEAR_COHERENCE_OUT)


=== KEYWORDS | TOP 10 MOST TIME-COHERENT ===
 rank  cluster_id  cluster_size  articles_with_year  median_year  avg_abs_dev_from_median  min_year  max_year
    1          44             4                   4       2014.0                    0.000    2014.0    2014.0
    2          45             4                   4       2008.0                    0.000    2008.0    2008.0
    3          61             3                   3       2002.0                    0.000    2002.0    2002.0
    4          56             3                   3       2022.0                    0.333    2021.0    2022.0
    5          57             3                   3       2025.0                    0.667    2023.0    2025.0
    6          41             4                   4       2001.5                    0.750    2000.0    2002.0
    7          36             5                   5       2020.0                    0.800    2018.0    2021.0
    8          54             3                   3       2005.0          

**Keyword cluster deep dive.** For each keyword cluster: extracts its most distinctive terms (TF-IDF style), most representative papers, bridge articles connecting it to other clusters, and year trends.

In [ ]:
# Deeper analysis for keyword clusters
# Outputs: distinctive terms, representative papers, bridge papers,
# time summaries, and an inter-cluster topic graph.


# Use the keyword graph from cell 0
keyword_graph = G

# Rebuild article_year if this cell is run without the year-deviation cell
if "article_year" not in globals():
    def _to_year_local(value):
        """Parse a year value from various string formats, returning an int or None."""
        if pd.isna(value):
            return None
        s = str(value).strip()
        if not s:
            return None
        digits = "".join(ch for ch in s if ch.isdigit())
        if len(digits) >= 4:
            y = int(digits[:4])
            if 1500 <= y <= 2100:
                return y
        try:
            y = int(float(s))
            if 1500 <= y <= 2100:
                return y
        except Exception:
            return None
        return None

    article_year = {}
    for _, row in meta.iterrows():
        article_id = normalize_id_from_pdf(row.get("pdf_file"))
        y_crossref = _to_year_local(row.get("crossref_year"))
        y_original = _to_year_local(row.get("original_year"))
        y_oa = _to_year_local(row.get("openalex_year"))
        article_year[article_id] = y_crossref if y_crossref is not None else (y_original if y_original is not None else y_oa)


# Helper maps
article_title = {}
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    article_title[article_id] = row["title"]

cluster_nodes = defaultdict(list)
for n, d in keyword_graph.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        cluster_nodes[cid].append(n)

# 1) Distinctive keywords per cluster using a simple lift-style score
cluster_kw_weight = defaultdict(lambda: defaultdict(float))
global_kw_weight = defaultdict(float)
cluster_total_kw_weight = defaultdict(float)
global_total_kw_weight = 0.0

for cid, nodes in cluster_nodes.items():
    for n in nodes:
        for kw, strength in article_kw_strength.get(n, {}).items():
            s = float(strength)
            cluster_kw_weight[cid][kw] += s
            global_kw_weight[kw] += s
            cluster_total_kw_weight[cid] += s
            global_total_kw_weight += s

distinctive_rows = []
cluster_topic_refined = {}
for cid, kw_map in cluster_kw_weight.items():
    rows = []
    c_total = cluster_total_kw_weight[cid]
    if c_total <= 0 or global_total_kw_weight <= 0:
        continue

    for kw, c_w in kw_map.items():
        g_w = global_kw_weight[kw]
        local_share = c_w / c_total
        global_share = g_w / global_total_kw_weight
        lift = local_share / global_share if global_share > 0 else 0.0
        score = c_w * math.log1p(lift)
        rows.append((kw, c_w, lift, score))

    rows = sorted(rows, key=lambda x: x[3], reverse=True)
    top_terms = rows[:10]
    cluster_topic_refined[cid] = ", ".join([kw for kw, _, _, _ in top_terms[:3]]) if top_terms else "unknown"

    for rank, (kw, c_w, lift, score) in enumerate(top_terms, start=1):
        distinctive_rows.append(
            {
                "cluster_id": cid,
                "rank": rank,
                "keyword": kw,
                "cluster_keyword_weight": round(c_w, 4),
                "lift_vs_global": round(lift, 4),
                "distinctiveness_score": round(score, 4),
                "cluster_size": len(cluster_nodes[cid]),
            }
        )

distinctive_df = pd.DataFrame(distinctive_rows).sort_values(["cluster_id", "rank"])
distinctive_df.to_csv(KW_DISTINCTIVE_OUT, index=False)
print("Distinctive keyword report saved to:", KW_DISTINCTIVE_OUT)

# 2) Representative/core papers per cluster based on internal weighted degree
representative_rows = []
for cid, nodes in cluster_nodes.items():
    node_set = set(nodes)
    scored = []
    for n in nodes:
        internal_strength = 0.0
        total_strength = 0.0
        for nbr, ed in keyword_graph[n].items():
            s = float(ed.get("strength", 1.0))
            total_strength += s
            if nbr in node_set:
                internal_strength += s
        purity = (internal_strength / total_strength) if total_strength > 0 else 0.0
        scored.append((n, internal_strength, total_strength, purity))

    scored = sorted(scored, key=lambda x: (x[1], x[3]), reverse=True)
    for rank, (n, internal_strength, total_strength, purity) in enumerate(scored[:10], start=1):
        representative_rows.append(
            {
                "cluster_id": cid,
                "rank": rank,
                "article_id": n,
                "title": article_title.get(n, n),
                "cluster_topic": cluster_topic_refined.get(cid, "unknown"),
                "internal_strength": round(internal_strength, 4),
                "total_strength": round(total_strength, 4),
                "internal_purity": round(purity, 4),
                "year": article_year.get(n) if "article_year" in globals() else None,
            }
        )

representative_df = pd.DataFrame(representative_rows).sort_values(["cluster_id", "rank"])
representative_df.to_csv(KW_REPRESENTATIVE_OUT, index=False)
print("Representative papers report saved to:", KW_REPRESENTATIVE_OUT)

# 3) Bridge papers between clusters
bridge_rows = []
for n in keyword_graph.nodes():
    cid = int(keyword_graph.nodes[n].get("cluster", -1))
    if cid < 0:
        continue

    internal_strength = 0.0
    external_strength = 0.0
    neighbor_clusters = set()

    for nbr, ed in keyword_graph[n].items():
        nbr_cid = int(keyword_graph.nodes[nbr].get("cluster", -1))
        s = float(ed.get("strength", 1.0))
        if nbr_cid == cid:
            internal_strength += s
        elif nbr_cid >= 0:
            external_strength += s
            neighbor_clusters.add(nbr_cid)

    total_strength = internal_strength + external_strength
    bridge_ratio = (external_strength / total_strength) if total_strength > 0 else 0.0
    if external_strength > 0:
        bridge_rows.append(
            {
                "article_id": n,
                "title": article_title.get(n, n),
                "cluster_id": cid,
                "cluster_topic": cluster_topic_refined.get(cid, "unknown"),
                "year": article_year.get(n) if "article_year" in globals() else None,
                "internal_strength": round(internal_strength, 4),
                "external_strength": round(external_strength, 4),
                "bridge_ratio": round(bridge_ratio, 4),
                "neighbor_cluster_count": len(neighbor_clusters),
            }
        )

bridge_df = pd.DataFrame(bridge_rows).sort_values(["bridge_ratio", "external_strength"], ascending=[False, False])
bridge_df.to_csv(KW_BRIDGES_OUT, index=False)
print("Bridge papers report saved to:", KW_BRIDGES_OUT)

# 4) Cluster time summaries with dominant period split
period_bins = [(-10**9, 1977, "pre-1977"), (1977, 1997, "1977-1997"), (1997, 2017, "1997-2017"), (2017, 10**9, "2017+")]

time_rows = []
for cid, nodes in cluster_nodes.items():
    years = [article_year.get(n) for n in nodes] if "article_year" in globals() else []
    years = [int(y) for y in years if y is not None]

    period_counts = defaultdict(int)
    for y in years:
        for lo, hi, label in period_bins:
            if lo <= y < hi:
                period_counts[label] += 1
                break

    dominant_period = None
    dominant_count = 0
    if period_counts:
        dominant_period, dominant_count = sorted(period_counts.items(), key=lambda x: x[1], reverse=True)[0]

    time_rows.append(
        {
            "cluster_id": cid,
            "cluster_topic": cluster_topic_refined.get(cid, "unknown"),
            "cluster_size": len(nodes),
            "articles_with_year": len(years),
            "earliest_year": min(years) if years else None,
            "latest_year": max(years) if years else None,
            "mean_year": round(sum(years) / len(years), 2) if years else None,
            "dominant_period": dominant_period,
            "dominant_period_share": round(dominant_count / len(years), 4) if years else None,
            "pre_1977": period_counts.get("pre-1977", 0),
            "1977_1997": period_counts.get("1977-1997", 0),
            "1997_2017": period_counts.get("1997-2017", 0),
            "2017_plus": period_counts.get("2017+", 0),
        }
    )

time_df = pd.DataFrame(time_rows).sort_values(["cluster_size", "cluster_id"], ascending=[False, True])
time_df.to_csv(KW_TIME_OUT, index=False)
print("Cluster time-trend report saved to:", KW_TIME_OUT)

# 5) Inter-cluster graph (topic map)
cluster_graph = nx.Graph()
for cid, nodes in cluster_nodes.items():
    cluster_graph.add_node(
        cid,
        label=f"C{cid}",
        title=f"C{cid}: {cluster_topic_refined.get(cid, 'unknown')}",
        cluster_size=len(nodes),
        cluster_topic=cluster_topic_refined.get(cid, "unknown"),
    )

inter_acc = defaultdict(float)
for u, v, ed in keyword_graph.edges(data=True):
    cu = int(keyword_graph.nodes[u].get("cluster", -1))
    cv = int(keyword_graph.nodes[v].get("cluster", -1))
    if cu < 0 or cv < 0 or cu == cv:
        continue
    key = (cu, cv) if cu < cv else (cv, cu)
    inter_acc[key] += float(ed.get("strength", 1.0))

for (cu, cv), s in inter_acc.items():
    cluster_graph.add_edge(cu, cv, strength=round(s, 4))

nx.write_graphml(cluster_graph, KW_CLUSTER_GRAPHML_OUT)
print("Inter-cluster graphml saved to:", KW_CLUSTER_GRAPHML_OUT)

net = Network(height="900px", width="100%", bgcolor="white", font_color="black", notebook=False)
net.barnes_hut(
    gravity=-22000,
    central_gravity=0.08,
    spring_length=170,
    spring_strength=0.02,
    damping=0.85,
    overlap=0.15,
)

for cid, nd in cluster_graph.nodes(data=True):
    net.add_node(
        cid,
        label=f"C{cid}",
        title=f"C{cid}<br>Topic: {nd.get('cluster_topic','unknown')}<br>Size: {nd.get('cluster_size',0)}",
        value=max(8, int(nd.get("cluster_size", 1))),
        color=color_for_cluster(int(cid)),
    )

for cu, cv, ed in cluster_graph.edges(data=True):
    s = float(ed.get("strength", 1.0))
    net.add_edge(
        cu,
        cv,
        width=0.8 + min(8.0, s / 20.0),
        color={"color": "#888888", "opacity": 0.35},
        title=f"inter-cluster strength={s:.3f}",
    )

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1200}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 120
  }
}
""")
net.write_html(KW_CLUSTER_HTML_OUT, open_browser=False)
print("Inter-cluster HTML saved to:", KW_CLUSTER_HTML_OUT)

# Console previews
print("\nTop distinctive terms preview:")
print(distinctive_df.head(15).to_string(index=False))

print("\nTop representative papers preview:")
print(representative_df.head(15).to_string(index=False))

print("\nTop bridge papers preview:")
print(bridge_df.head(15).to_string(index=False))

print("\nCluster time summary preview:")
print(time_df.head(15).to_string(index=False))

Distinctive keyword report saved to: /home/john/repos/leno4ka/output/keyword_cluster_distinctive_terms.csv
Representative papers report saved to: /home/john/repos/leno4ka/output/keyword_cluster_representative_papers.csv
Bridge papers report saved to: /home/john/repos/leno4ka/output/keyword_cluster_bridge_articles.csv
Cluster time-trend report saved to: /home/john/repos/leno4ka/output/keyword_cluster_time_trends.csv
Inter-cluster graphml saved to: /home/john/repos/leno4ka/output/keyword_cluster_intercluster.graphml
Inter-cluster HTML saved to: /home/john/repos/leno4ka/output/keyword_cluster_intercluster.html

Top distinctive terms preview:
 cluster_id  rank                       keyword  cluster_keyword_weight  lift_vs_global  distinctiveness_score  cluster_size
          0     1                 work stealing                 64.2500         25.3177               210.1131            54
          0     2                load balancing                 63.6333         21.3980               1

## Practical interpretation
Authors view: best for identifying research generations / publication eras.

Keywords view: best for identifying topical waves.

Combined view: best for broad map overview, not for time-based conclusions.

In [161]:
# Clustering by publication year periods + year distributions and mean year visualization

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})



# Reuse article_year from previous cell if present; otherwise rebuild safely
if "article_year" not in globals():
    def _to_year(value):
        """Parse a year value from various string formats, returning an int or None."""
        if pd.isna(value):
            return None
        s = str(value).strip()
        if not s:
            return None
        digits = "".join(ch for ch in s if ch.isdigit())
        if len(digits) >= 4:
            y = int(digits[:4])
            if 1500 <= y <= 2100:
                return y
        try:
            y = int(float(s))
            if 1500 <= y <= 2100:
                return y
        except Exception:
            return None
        return None

    article_year = {}
    for _, row in meta.iterrows():
        article_id = normalize_id_from_pdf(row.get("pdf_file"))
        y_crossref = _to_year(row.get("crossref_year"))
        y_original = _to_year(row.get("original_year"))
        y_oa = _to_year(row.get("openalex_year"))
        article_year[article_id] = y_crossref if y_crossref is not None else (y_original if y_original is not None else y_oa)


# Build article-year frame
rows = []
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    rows.append(
        {
            "article_id": article_id,
            "title": row["title"],
            "year": article_year.get(article_id),
        }
    )

year_df = pd.DataFrame(rows)


# Period bins (interpreting user ranges)
# 1950-1977, 1977-1997, 1997-2017, 2017+
# left-inclusive, right-exclusive except last bin

def year_period(y) -> str:
    """Map a year integer to a named period string."""
    if pd.isna(y):
        return "unknown_year"
    y = int(y)
    if 1950 <= y < 1977:
        return "1950-1977"
    if 1977 <= y < 1997:
        return "1977-1997"
    if 1997 <= y < 2017:
        return "1997-2017"
    if y >= 2017:
        return "2017+"
    return "before_1950"


year_df["period"] = year_df["year"].map(year_period)

unknown_year_count = int((year_df["period"] == "unknown_year").sum())
before_1950_count = int((year_df["period"] == "before_1950").sum())
print("unknown year articles:", unknown_year_count)
print("before_1950 articles:", before_1950_count)

# Keep requested periods only
period_order = ["1950-1977", "1977-1997", "1997-2017", "2017+"]
year_df = year_df[year_df["period"].isin(period_order)].copy()
year_df["year"] = year_df["year"].astype(int)


# Distribution by years inside each period
period_year_dist = (
    year_df.groupby(["period", "year"], as_index=False)
    .size()
    .rename(columns={"size": "article_count"})
    .sort_values(["period", "year"])
)

# Summary per period (mean year + spread)
period_summary = (
    year_df.groupby("period", as_index=False)
    .agg(
        article_count=("article_id", "count"),
        mean_year=("year", "mean"),
        median_year=("year", "median"),
        std_year=("year", "std"),
        min_year=("year", "min"),
        max_year=("year", "max"),
    )
)
period_summary["period"] = pd.Categorical(period_summary["period"], categories=period_order, ordered=True)
period_summary = period_summary.sort_values("period")
period_summary["mean_year"] = period_summary["mean_year"].round(2)
period_summary["median_year"] = period_summary["median_year"].round(2)
period_summary["std_year"] = period_summary["std_year"].round(3)


# Save outputs
period_year_dist.to_csv(YEAR_PERIOD_DIST_OUT, index=False)
period_summary.to_csv(YEAR_PERIOD_SUMMARY_OUT, index=False)

print("Year-period distribution saved to:", YEAR_PERIOD_DIST_OUT)
print("Year-period summary saved to:", YEAR_PERIOD_SUMMARY_OUT)
print(period_summary.to_string(index=False))


# Visualization: one subplot per period with mean-year marker
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes = axes.flatten()

for i, period in enumerate(period_order):
    ax = axes[i]
    d = period_year_dist[period_year_dist["period"] == period]
    s = period_summary[period_summary["period"] == period]

    if d.empty:
        ax.set_title(f"{period} (no data)")
        ax.set_xlabel("Year")
        ax.set_ylabel("Article count")
        continue

    ax.bar(d["year"], d["article_count"], color="#4e79a7", alpha=0.85)

    mean_year = float(s["mean_year"].iloc[0]) if not s.empty else None
    if mean_year is not None:
        ax.axvline(mean_year, color="#e15759", linestyle="--", linewidth=2, label=f"mean={mean_year:.2f}")

    ax.set_title(f"{period} | n={int(s['article_count'].iloc[0]) if not s.empty else len(d)}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Article count")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(YEAR_PERIOD_PLOT_OUT, dpi=180)

print("Year-period visualization saved to:", YEAR_PERIOD_PLOT_OUT)

unknown year articles: 409
before_1950 articles: 12
Year-period distribution saved to: /home/john/repos/leno4ka/output/articles_year_period_distribution.csv
Year-period summary saved to: /home/john/repos/leno4ka/output/articles_year_period_summary.csv
   period  article_count  mean_year  median_year  std_year  min_year  max_year
1950-1977             46    1967.54       1970.0     7.351      1950      1976
1977-1997            248    1988.19       1990.0     5.707      1977      1996
1997-2017           1106    2007.14       2008.0     4.989      1997      2016
    2017+            555    2021.10       2021.0     2.513      2017      2026
Year-period visualization saved to: /home/john/repos/leno4ka/output/articles_year_period_distribution.png


**Cluster similarity matrix.** Computes pairwise cosine similarity between clusters based on their keyword weight profiles and visualises the result as a heatmap.

In [162]:
# Cluster-to-cluster cosine similarity based on keyword weight profiles

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

keyword_graph = G  # keyword view from cell 0

# Build cluster keyword weight vectors
cluster_kw = {}
for n, d in keyword_graph.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid < 0:
        continue
    for kw, strength in article_kw_strength.get(n, {}).items():
        cluster_kw.setdefault(cid, {})
        cluster_kw[cid][kw] = cluster_kw[cid].get(kw, 0.0) + float(strength)

cluster_ids = sorted(cluster_kw.keys())
all_kws = sorted({kw for v in cluster_kw.values() for kw in v})

def cosine(a, b, vocab):
    """Compute cosine similarity between two keyword weight dicts over a shared vocabulary."""
    dot = sum(a.get(k, 0.0) * b.get(k, 0.0) for k in vocab)
    na = math.sqrt(sum(v**2 for v in a.values()))
    nb = math.sqrt(sum(v**2 for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

sim_matrix = pd.DataFrame(index=cluster_ids, columns=cluster_ids, dtype=float)
for ci in cluster_ids:
    for cj in cluster_ids:
        sim_matrix.loc[ci, cj] = cosine(cluster_kw[ci], cluster_kw[cj], all_kws)

# Only show clusters larger than a threshold to keep the plot readable
min_size = 5
large_cids = [cid for cid in cluster_ids if len([n for n, d in keyword_graph.nodes(data=True) if int(d.get("cluster", -1)) == cid]) >= min_size]
sub = sim_matrix.loc[large_cids, large_cids].astype(float)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(sub.values, vmin=0, vmax=1, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_xticks(range(len(large_cids)))
ax.set_yticks(range(len(large_cids)))
ax.set_xticklabels([f"C{c}" for c in large_cids], rotation=90, fontsize=7)
ax.set_yticklabels([f"C{c}" for c in large_cids], fontsize=7)
ax.set_title("Cluster keyword-profile cosine similarity (clusters ≥ 5 nodes)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cluster_cosine_similarity.png", dpi=160)

# Print top-5 most similar cluster pairs (excluding self)
pairs = [(sim_matrix.loc[ci, cj], ci, cj)
         for i, ci in enumerate(cluster_ids)
         for cj in cluster_ids[i+1:]]
pairs.sort(reverse=True)
print("Top 10 most similar cluster pairs:")
for sim, ci, cj in pairs[:10]:
    print(f"  C{ci} <-> C{cj}: {sim:.4f}")


Top 10 most similar cluster pairs:
  C29 <-> C42: 0.2832
  C19 <-> C54: 0.2697
  C11 <-> C93: 0.2514
  C37 <-> C46: 0.2460
  C34 <-> C63: 0.1960
  C9 <-> C110: 0.1809
  C6 <-> C111: 0.1703
  C2 <-> C20: 0.1680
  C20 <-> C21: 0.1679
  C104 <-> C123: 0.1648


**Keyword prominence over time.** Tracks how often each keyword appears per decade to reveal emerging, declining, and stable research themes across the corpus.

In [163]:
# Keyword prominence by decade - shows emerging vs. legacy topics
# Self-contained: loads data from CSVs directly.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})



def normalize_id_from_pdf(pdf_name):
    """Normalize article ID from a PDF filename."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    if s.lower().endswith(".pdf"):
        s = s[:-4]
    return s.lower()


def normalize_id_from_filename(filename):
    """Normalize article ID from a plain filename."""
    if pd.isna(filename):
        return ""
    return str(filename).strip().lower()


def parse_keywords_with_strength(cell):
    """Parse keyword-strength pairs from a raw cell string."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
    except Exception:
        return []
    out = []
    for item in parsed:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            kw = str(item[0]).strip().lower()
            try:
                strength = float(item[1])
            except Exception:
                continue
            if kw:
                out.append((kw, strength))
    return out


def to_year(value):
    """Parse a publication year from various string formats."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    return None


# Load data
meta = pd.read_csv(META_PATH)
kw = pd.read_csv(KW_PATH)
meta["article_id"] = meta["pdf_file"].map(normalize_id_from_pdf)
kw["article_id"] = kw["filename"].map(normalize_id_from_filename)
kw["keyword_items"] = kw["keywords_without_stopwords"].map(parse_keywords_with_strength)

# Build article_year
article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    y = to_year(row.get("crossref_year")) or to_year(row.get("original_year")) or to_year(row.get("openalex_year"))
    article_year[aid] = y

# Build article_kw_strength
article_kw_strength = {}
for _, row in kw.iterrows():
    aid = row["article_id"]
    items = row["keyword_items"]
    if not items:
        continue
    tmp = defaultdict(list)
    for kw_term, kw_strength in items:
        tmp[kw_term].append(float(kw_strength))
    article_kw_strength[aid] = {k: sum(v) / len(v) for k, v in tmp.items()}

# Collect rows + track articles per decade
rows = []
decade_article_ids = defaultdict(set)
for aid, y in article_year.items():
    if y is None:
        continue
    decade = (int(y) // 10) * 10
    decade_article_ids[decade].add(aid)
    for kw_term, strength in article_kw_strength.get(aid, {}).items():
        rows.append({"decade": decade, "keyword": kw_term, "strength": float(strength)})

kw_decade_df = pd.DataFrame(rows)
articles_per_decade = pd.Series({d: len(ids) for d, ids in decade_article_ids.items()})

# Top-40 keywords by number of articles containing them (most widespread topics)
global_top_kws = (
    kw_decade_df.groupby("keyword")["decade"].count()
    .sort_values(ascending=False)
    .head(40)
    .index.tolist()
)

pivot = (
    kw_decade_df[kw_decade_df["keyword"].isin(global_top_kws)]
    .groupby(["keyword", "decade"])["strength"]
    .sum()
    .unstack(fill_value=0)
)

# Normalize by articles per decade -> absolute per-article keyword density
pivot_density = pivot.div(articles_per_decade, axis=1).fillna(0)

# Sort keywords by their peak decade (left = oldest peak, right = newest)
peak_decade = pivot_density.idxmax(axis=1)
pivot_density = pivot_density.loc[peak_decade.sort_values().index]

# Log scale: replace zeros with NaN so they render as black
data = pivot_density.values.copy()
data[data == 0] = np.nan

vmin = np.nanmin(data[data > 0])
vmax = np.nanmax(data)
norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

fig, ax = plt.subplots(figsize=(16, 11))
cmap = plt.cm.plasma.copy()
cmap.set_bad("#111111")  # zeros -> near-black
im = ax.imshow(data, aspect="auto", cmap=cmap, norm=norm)
plt.colorbar(im, ax=ax, label="Avg keyword strength per article (log scale)")
ax.set_xticks(range(len(pivot_density.columns)))
ax.set_xticklabels([str(d) for d in pivot_density.columns], rotation=45, ha="right")
ax.set_yticks(range(len(pivot_density.index)))
ax.set_yticklabels(pivot_density.index, fontsize=8)
ax.set_title("Top-40 keywords: absolute prominence per article by decade (log scale, sorted by peak decade)")
plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=160)
print(f"Saved to {OUTPUT_PATH}")


Saved to /home/john/repos/leno4ka/output/keyword_temporal_heatmap.png


**Decade champions.** Lists the top keywords for each decade independently, showing which terms dominated each era rather than the corpus as a whole.

In [164]:
# Option 1: per-decade top keywords — each era's own champions
# Requires: pivot_density, articles_per_decade from cell 14

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

TOP_PER_DECADE = 10  # how many top keywords to pick from each decade

# For each decade pick top keywords by per-article density
era_top_kws = set()
for decade_col in pivot_density.columns:
    top = pivot_density[decade_col].sort_values(ascending=False).head(TOP_PER_DECADE).index
    era_top_kws.update(top)

era_pivot = pivot_density.loc[sorted(era_top_kws)]

# Sort rows by peak decade
peak_decade = era_pivot.idxmax(axis=1)
era_pivot = era_pivot.loc[peak_decade.sort_values().index]

data = era_pivot.values.copy().astype(float)
data[data == 0] = np.nan

vmin = np.nanmin(data[data > 0])
vmax = np.nanmax(data)
norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
cmap = plt.cm.plasma.copy()
cmap.set_bad("#111111")

fig, ax = plt.subplots(figsize=(16, max(8, len(era_pivot) * 0.28)))
im = ax.imshow(data, aspect="auto", cmap=cmap, norm=norm)
plt.colorbar(im, ax=ax, label="Avg keyword strength per article (log scale)")
ax.set_xticks(range(len(era_pivot.columns)))
ax.set_xticklabels([str(d) for d in era_pivot.columns], rotation=45, ha="right")
ax.set_yticks(range(len(era_pivot.index)))
ax.set_yticklabels(era_pivot.index, fontsize=8)
ax.set_title(f"Keywords that defined each era (top-{TOP_PER_DECADE} per decade, sorted by peak decade)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "keyword_temporal_heatmap_era.png", dpi=160)
print(f"Total unique keywords shown: {len(era_pivot)}")


Total unique keywords shown: 37


**Cross-cluster authors.** Identifies researchers who publish across multiple keyword clusters, acting as conceptual bridges between distinct research communities.

In [ ]:
# Authors who publish across multiple keyword clusters
keyword_graph = G

node_cluster = {n: int(d.get("cluster", -1)) for n, d in keyword_graph.nodes(data=True)}

author_clusters = defaultdict(set)
author_articles = defaultdict(list)

for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    cid = node_cluster.get(article_id, -1)
    if cid < 0:
        continue
    for author in row["authors"]:
        author = author.strip()
        if author:
            author_clusters[author].add(cid)
            author_articles[author].append(article_id)

cross_rows = []
for author, clusters in author_clusters.items():
    if len(clusters) >= 2:
        cross_rows.append({
            "author": author,
            "n_clusters": len(clusters),
            "cluster_ids": "|".join(str(c) for c in sorted(clusters)),
            "n_articles": len(author_articles[author]),
        })

cross_df = pd.DataFrame(cross_rows).sort_values(["n_clusters", "n_articles"], ascending=False)
print(f"Authors spanning ≥2 keyword clusters: {len(cross_df)}")
print(cross_df.head(25).to_string(index=False))

cross_df.to_csv(OUTPUT_DIR / "author_cross_cluster_membership.csv", index=False)


Authors spanning ≥2 keyword clusters: 131
              author  n_clusters                                                          cluster_ids  n_articles
             Nan Nan          25 0|1|2|3|4|5|8|9|10|12|15|16|18|23|25|26|27|30|40|42|71|82|87|112|122          45
       Fernand Gobet           8                                                3|4|5|16|25|37|46|109          39
          Matej Guid           8                                              7|14|15|21|22|52|94|114          22
    Paolo Ciancarini           8                                                  0|2|6|7|9|14|21|118          19
         Ivan Bratko           7                                                  0|7|12|14|23|52|114          19
  Jonathan Schaeffer           6                                                      0|7|10|12|13|24          15
       Merim Bilalić           6                                                     3|16|25|34|46|50          11
Guillermo Campitelli           5              

**Structural importance via betweenness.** Ranks papers by betweenness centrality — high-centrality papers sit on many shortest paths and act as connectors between remote parts of the network.

In [ ]:
# Betweenness centrality to rank papers by structural importance
keyword_graph = G

# Approximate betweenness (k=100 is enough for ranking on large graphs)
n = keyword_graph.number_of_nodes()
k = min(200, n)
bc = nx.betweenness_centrality(keyword_graph, k=k, weight="strength", normalized=True, seed=42)
dc = dict(keyword_graph.degree(weight="strength"))

centrality_rows = []
for n_id, score in bc.items():
    d = keyword_graph.nodes[n_id]
    cid = int(d.get("cluster", -1))
    centrality_rows.append({
        "article_id": n_id,
        "title": d.get("title", n_id),
        "cluster_id": cid,
        "cluster_topic": d.get("cluster_topic", "unknown"),
        "betweenness": round(score, 6),
        "weighted_degree": round(dc.get(n_id, 0.0), 4),
        "year": article_year.get(n_id),
    })

centrality_df = pd.DataFrame(centrality_rows).sort_values("betweenness", ascending=False)
centrality_df.to_csv(OUTPUT_DIR / "article_betweenness_centrality.csv", index=False)

print("Top 20 most central articles (structural bridges):")
print(centrality_df[["title", "cluster_id", "cluster_topic", "betweenness", "year"]].head(20).to_string(index=False))


Top 20 most central articles (structural bridges):
                                                                                    title  cluster_id                                        cluster_topic  betweenness   year
                     3 Establishing knowledge spaces by systematical problem construction           3     turing machine, problem space, production system     0.017355    NaN
                                                     A FIVE-YEAR PLAN FOR AUTOMATIC CHESS          15       gender differences, risk taking, risk aversion     0.015956 2013.0
                            Playing "Invisible Chess" with Information-Theoretic Advisors           7      proof-number search, pn search, zobrist hashing     0.015303 2001.0
                                                      Games solved: Now and in the future           7      proof-number search, pn search, zobrist hashing     0.012698 2002.0
                                                       Designing an AI age

**RQ4-A — Era-coloured graph.** Colours every node by publication era instead of Leiden cluster to reveal whether temporal periods form spatially coherent regions in the keyword graph.

In [ ]:
# ---------------------------------------------------------------------------
# RQ4-A — Era-colored Pyvis visualization
#
# Same keyword-similarity graph from Cell 0, but node color = publication era
# instead of Leiden cluster. This shows WHERE in the network each era lives
# and whether temporal periods form spatially coherent regions or are mixed.
# ---------------------------------------------------------------------------


ERA_COLORS = {
    "1950-1977":   "#3498db",
    "1977-1997":   "#1abc9c",
    "1997-2017":   "#f39c12",
    "2017+":       "#e74c3c",
    "before_1950": "#9b59b6",
    "unknown":     "#7f8c8d",
}

def get_era_rq4(year) -> str:
    """Map a year value to an era label for RQ4 analysis."""
    if pd.isna(year) or str(year).strip().lower() in ("", "nan"):
        return "unknown"
    try:
        y = int(float(str(year).strip()))
    except (ValueError, TypeError):
        return "unknown"
    if 1950 <= y < 1977:
        return "1950-1977"
    if 1977 <= y < 1997:
        return "1977-1997"
    if 1997 <= y < 2017:
        return "1997-2017"
    if y >= 2017:
        return "2017+"
    return "before_1950"

meta_new = pd.read_csv(META_PATH)
def _norm_pdf(x):
    """Normalize a PDF filename to a lowercase ID."""
    if pd.isna(x): 
        return ""
    s = str(x).strip()
    return (s[:-4] if s.lower().endswith(".pdf") else s).lower()

meta_new["article_id"] = meta_new["pdf_file"].map(_norm_pdf)
aid_to_era = {}
for _, row in meta_new.iterrows():
    y = row.get("crossref_year")
    if pd.isna(y) or str(y).strip() == "":
        y = row.get("original_year")
    if pd.isna(y) or str(y).strip() == "":
        y = row.get("openalex_year")
    aid_to_era[row["article_id"]] = get_era_rq4(y)

keyword_graph = G  # from Cell 0
connected = [n for n, d in keyword_graph.degree() if d > 0]
G_vis = keyword_graph.subgraph(connected).copy()

net = Network(height="950px", width="100%", bgcolor="white", font_color="black",
              notebook=False, directed=False)
net.barnes_hut(gravity=-42000, central_gravity=0.06, spring_length=260,
               spring_strength=0.012, damping=0.86, overlap=0.2)

for n, nd in G_vis.nodes(data=True):
    era   = aid_to_era.get(n, "unknown")
    cid   = int(nd.get("cluster", -1))
    color = ERA_COLORS.get(era, "#7f8c8d")
    net.add_node(
        n,
        label=era,
        group=era,
        color=color,
        value=8 + G_vis.degree(n),
        title=(
            f"<b>{nd.get('title', n)}</b><br>"
            f"Era: {era}<br>"
            f"Keyword cluster: C{cid}<br>"
            f"Degree: {G_vis.degree(n)}"
        ),
    )

for u, v, ed in G_vis.edges(data=True):
    era_u = aid_to_era.get(u, "unknown")
    era_v = aid_to_era.get(v, "unknown")
    same_era = era_u == era_v
    net.add_edge(u, v,
                 width=1.2 if same_era else 0.3,
                 color={"color": ERA_COLORS.get(era_u, "#666"), "opacity": 0.6 if same_era else 0.1})

net.set_options("""
var options = {
  "physics": {"enabled": true, "solver": "barnesHut",
               "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}},
  "nodes": {"font": {"size": 11}},
  "interaction": {"hover": true, "tooltipDelay": 150}
}
""")
net.write_html(ERA_HTML_OUT, open_browser=False)

era_counts = {}
for n in connected:
    e = aid_to_era.get(n, "unknown")
    era_counts[e] = era_counts.get(e, 0) + 1
print(f"Era-colored visualization -> {ERA_HTML_OUT}")
print("Era distribution in connected graph:", dict(sorted(era_counts.items())))

Era-colored visualization -> /home/john/repos/leno4ka/output/articles_leiden_era_colored.html
Era distribution in connected graph: {'1950-1977': 17, '1977-1997': 121, '1997-2017': 558, '2017+': 303, 'before_1950': 2, 'unknown': 176}


**RQ4-B — Keyword evolution.** Classifies keywords as emerging, declining, or stable based on their decade-by-decade frequency trend and writes the classification to CSV.

In [168]:
# ---------------------------------------------------------------------------
# RQ4-B — Emerging / declining / stable keyword classification
#          + cross-era citation flow heatmap
#
# Uses keyword_emergence.csv and cross_era_flow.csv from skg_metrix.ipynb.
# ---------------------------------------------------------------------------
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})


ERAS = ["1950-1977", "1977-1997", "1997-2017", "2017+"]

emg = pd.read_csv(KW_EMG_PATH)

# Classify each keyword
def classify_keyword(row):
    """Classify a keyword as emerging, declining, or stable based on era counts."""
    strengths = [row.get(f"strength_{e}", 0) for e in ERAS]
    nonzero = [s for s in strengths if s > 0]
    if len(nonzero) == 0:
        return "absent"
    peak_idx = ERAS.index(row["peak_era"]) if row["peak_era"] in ERAS else -1
    first_idx = ERAS.index(row["first_era"]) if row["first_era"] in ERAS else -1
    last_strength = row.get("strength_2017+", 0)
    first_strength = strengths[first_idx] if first_idx >= 0 else 0
    growth = row.get("growth_rate", 0)

    # Emerging: appeared recently OR strong growth into 2017+
    if first_idx >= 2 or growth > 5.0:
        return "emerging"
    # Declining: peaked early, weak/zero in 2017+
    if peak_idx <= 1 and last_strength < first_strength * 0.3:
        return "declining"
    # Stable: present across most eras without strong trend
    if len(nonzero) >= 3:
        return "stable"
    return "sporadic"

emg["category"] = emg.apply(classify_keyword, axis=1)
emg.to_csv(KW_CLASS_OUT, index=False)

print("Keyword category counts:")
print(emg["category"].value_counts().to_string())

for cat in ("emerging", "declining", "stable"):
    subset = emg[emg["category"] == cat].sort_values("growth_rate", ascending=(cat == "declining"))
    print(f"\nTop 10 {cat} keywords:")
    print(subset[["keyword","first_era","peak_era","growth_rate"]].head(10).to_string(index=False))

# ── Cross-era citation flow heatmap ───────────────────────────────────────────
flow_df = pd.read_csv(ERA_FLOW_PATH)
pivot = flow_df.pivot(index="citing_era", columns="cited_era", values="edge_count").fillna(0)
pivot = pivot.reindex(index=ERAS, columns=ERAS, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw counts heatmap
ax = axes[0]
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax, label="Citation edges")
ax.set_xticks(range(len(ERAS))); ax.set_xticklabels(ERAS, rotation=30, ha="right")
ax.set_yticks(range(len(ERAS))); ax.set_yticklabels(ERAS)
ax.set_xlabel("Cited era"); ax.set_ylabel("Citing era")
ax.set_title("Cross-era citation flow (raw counts)")
for i in range(len(ERAS)):
    for j in range(len(ERAS)):
        ax.text(j, i, int(pivot.values[i, j]), ha="center", va="center", fontsize=9,
                color="white" if pivot.values[i, j] > pivot.values.max() * 0.5 else "black")

# Row-normalized (what % of each era's outgoing citations go to each era)
pivot_norm = pivot.div(pivot.sum(axis=1) + 1e-9, axis=0)
ax2 = axes[1]
im2 = ax2.imshow(pivot_norm.values, cmap="Blues", aspect="auto", vmin=0, vmax=1)
plt.colorbar(im2, ax=ax2, label="Fraction of outgoing citations")
ax2.set_xticks(range(len(ERAS))); ax2.set_xticklabels(ERAS, rotation=30, ha="right")
ax2.set_yticks(range(len(ERAS))); ax2.set_yticklabels(ERAS)
ax2.set_xlabel("Cited era"); ax2.set_ylabel("Citing era")
ax2.set_title("Cross-era citation flow (row-normalised)")
for i in range(len(ERAS)):
    for j in range(len(ERAS)):
        ax2.text(j, i, f"{pivot_norm.values[i,j]:.2f}", ha="center", va="center", fontsize=9,
                 color="white" if pivot_norm.values[i, j] > 0.5 else "black")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rq4_cross_era_flow.png", dpi=150)
print("Saved -> rq4_cross_era_flow.png")

Keyword category counts:
category
emerging     23502
declining     4268
stable         937
sporadic       793

Top 10 emerging keywords:
              keyword first_era  peak_era  growth_rate
       adam optimizer 1997-2017     2017+      29.5000
  cognitive processes 1950-1977 1997-2017      25.0000
               komodo 1997-2017     2017+      19.5000
      stochastic game 1977-1997     2017+      18.9167
      time management 1950-1977     2017+      17.7250
cognitive performance 1977-1997     2017+      16.0595
          blitz chess 1977-1997     2017+      15.6250
             win rate 1997-2017     2017+      15.5000
          overfitting 1977-1997     2017+      15.5000
   mean squared error 1997-2017     2017+      15.5000

Top 10 declining keywords:
               keyword first_era  peak_era  growth_rate
        dataflow graph 1977-1997 1977-1997          0.0
                dtim e 1977-1997 1977-1997          0.0
           undecidable 1977-1997 1977-1997          0.0
     a

**Year-coloured graph.** Colours each node on a red-to-green gradient from oldest to newest publication, giving a direct visual read of where recent work sits in the network.

In [ ]:
# Year-colored visualization of articles_leiden_clusters
# Nodes colored red (oldest) -> green (youngest); unknown-year articles excluded.
# Color uses percentile rank so the full gradient is always evenly distributed.

def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        y = int(float(s))
        if 1500 <= y <= 2100:
            return y
    except Exception:
        return None
    return None


# Build article_id -> year mapping (reuses meta from Cell 0)
article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    y = _to_year(row.get("crossref_year"))
    if y is None:
        y = _to_year(row.get("original_year"))
    if y is None:
        y = _to_year(row.get("openalex_year"))
    article_year[aid] = y


def rank_to_color(rank):
    """rank in [0, 1]: 0 = oldest (red), 1 = newest (green)."""
    hue = rank * (120 / 360)
    r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 0.95)
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))



# Keep only clustered nodes whose year is known
known_nodes = [
    n for n in G.nodes()
    if int(G.nodes[n].get("cluster", -1)) >= 0 and article_year.get(n) is not None
]
G_year = G.subgraph(known_nodes).copy()

# Compute percentile rank for each node's year
node_list = list(G_year.nodes())
years_sorted = sorted(set(article_year[n] for n in node_list))
year_rank = {y: i / (len(years_sorted) - 1) for i, y in enumerate(years_sorted)} if len(years_sorted) > 1 else {years_sorted[0]: 0.5}

y_min, y_max = years_sorted[0], years_sorted[-1]
print(f"Year range: {y_min} – {y_max}  |  unique years: {len(years_sorted)}  |  nodes: {G_year.number_of_nodes()}  |  edges: {G_year.number_of_edges()}")

net = Network(height="950px", width="100%", bgcolor="white", font_color="black", notebook=False)
net.barnes_hut(
    gravity=-42000,
    central_gravity=0.06,
    spring_length=260,
    spring_strength=0.012,
    damping=0.86,
    overlap=0.2,
)

for n, nd in G_year.nodes(data=True):
    year = article_year[n]
    color = rank_to_color(year_rank[year])
    cluster_id = int(nd.get("cluster", -1))
    topic_label = nd.get("cluster_topic", "unknown")
    node_size = 8 + min(24, G_year.degree(n))
    net.add_node(
        n,
        label=str(year),
        title=(
            f"{nd.get('title', n)}"
            f"<br>Year: {year}"
            f"<br>Cluster: C{cluster_id}"
            f"<br>Topic: {topic_label}"
            f"<br>Degree: {G_year.degree(n)}"
            f"<br>Authors: {nd.get('authors', '')}"
        ),
        value=node_size,
        color=color,
    )

for u, v, ed in G_year.edges(data=True):
    strength = float(ed.get("strength", 1.0))
    net.add_edge(
        u, v,
        width=0.6 + 0.3 * max(0.0, strength - 2.0),
        color={"color": "#888888", "opacity": 0.25},
        title=f"score={strength:.3f}",
    )

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "solver": "barnesHut",
    "stabilization": {"enabled": true, "iterations": 1800, "updateInterval": 25}
  },
  "nodes": {"font": {"size": 11}},
  "interaction": {"hover": true, "tooltipDelay": 150}
}
""")

net.write_html(YEAR_HTML_OUT, open_browser=False)
print("HTML written to:", YEAR_HTML_OUT)


Year range: 1938 – 2026  |  unique years: 62  |  nodes: 1001  |  edges: 1337
HTML written to: /home/john/repos/leno4ka/output/articles_leiden_year_colored.html


**INSIGHT 1 — Cluster size distribution.** Bar chart of article counts per cluster, revealing the core-periphery structure: a few dominant clusters and many small specialist ones.

In [170]:
# INSIGHT 1 — Cluster size distribution: dense core vs long tail
# Shows the "island archipelago" structure: a few large thematic hubs
# surrounded by dozens of micro-clusters (size 2-3).

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

cluster_sizes = sorted(
    [(int(d.get("cluster", -1)), G.degree(n))
     for n, d in G.nodes(data=True)
     if int(d.get("cluster", -1)) >= 0],
)
size_map = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        size_map[cid] = size_map.get(cid, 0) + 1

sizes = sorted(size_map.values(), reverse=True)
cids_sorted = sorted(size_map, key=lambda c: -size_map[c])

colors = ["#e74c3c" if sizes[i] >= 40
          else "#f39c12" if sizes[i] >= 15
          else "#3498db"
          for i in range(len(sizes))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cluster size distribution", fontsize=13, fontweight="bold")

# Bar chart
axes[0].bar(range(len(sizes)), sizes, color=colors, width=1.0, edgecolor="none")
axes[0].set_xlabel("Cluster rank (largest → smallest)")
axes[0].set_ylabel("Number of articles")
axes[0].set_title("All clusters")
axes[0].axhline(y=40, color="#e74c3c", linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].axhline(y=15, color="#f39c12", linestyle="--", linewidth=0.8, alpha=0.6)
legend = [
    mpatches.Patch(color="#e74c3c", label=f"Large (≥40): {sum(1 for s in sizes if s>=40)} clusters"),
    mpatches.Patch(color="#f39c12", label=f"Medium (15-39): {sum(1 for s in sizes if 15<=s<40)} clusters"),
    mpatches.Patch(color="#3498db", label=f"Micro (<15): {sum(1 for s in sizes if s<15)} clusters"),
]
axes[0].legend(handles=legend, fontsize=9)

# Cumulative coverage
cumulative = np.cumsum(sizes)
total = cumulative[-1]
axes[1].plot(range(1, len(sizes)+1), cumulative / total * 100, color="#2ecc71", linewidth=2)
axes[1].axhline(y=50, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Number of clusters (ranked)")
axes[1].set_ylabel("% of all connected articles covered")
axes[1].set_title("Cumulative coverage")
# Mark how many clusters cover 50% and 80%
for pct in [50, 80]:
    idx = next(i for i, v in enumerate(cumulative / total * 100) if v >= pct)
    axes[1].axvline(x=idx+1, color="orange", linestyle=":", linewidth=1)
    axes[1].text(idx+2, pct-4, f"{idx+1} clusters\ncover {pct}%", fontsize=8, color="orange")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_cluster_size_distribution.png", dpi=150, bbox_inches="tight")

micro = sum(1 for s in sizes if s < 15)
medium = sum(1 for s in sizes if 15 <= s < 40)
large = sum(1 for s in sizes if s >= 40)
isolated = size_map.get(-1, sum(1 for n, d in G.nodes(data=True) if int(d.get("cluster",-1)) == -1))
print(f"Large clusters  (≥40 articles): {large}")
print(f"Medium clusters (15–39):        {medium}")
print(f"Micro clusters  (<15):          {micro}")
print(f"Isolated nodes (no edges):      {len([n for n,d in G.nodes(data=True) if int(d.get('cluster',-1))==-1])}")
print(f"Top 5 clusters cover {cumulative[4]/total*100:.1f}% of all connected articles")
idx50 = next(i for i,v in enumerate(cumulative/total*100) if v>=50)
print(f"{idx50+1} clusters needed to cover 50% of connected articles")

Large clusters  (≥40 articles): 11
Medium clusters (15–39):        13
Micro clusters  (<15):          101
Isolated nodes (no edges):      1199
Top 5 clusters cover 20.6% of all connected articles
14 clusters needed to cover 50% of connected articles


**INSIGHT 2 — Inter-cluster edge density.** Counts cross-cluster edges to test whether the three main paradigms (cognitive science, game AI, computer chess) form isolated research silos.

In [171]:
# INSIGHT 2 — Inter-cluster edge density: three isolated paradigms
# If the three dominant clusters (cognitive science, parallel search, deep RL)
# are truly isolated, the heatmap will show near-zero off-diagonal cells
# for those cluster pairs.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

TOP_N = 12  # show top N clusters by size

size_map = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        size_map[cid] = size_map.get(cid, 0) + 1

top_clusters = sorted(size_map, key=lambda c: -size_map[c])[:TOP_N]

# Count edges between every pair of top clusters
edge_counts = {(a, b): 0 for a in top_clusters for b in top_clusters}
for u, v in G.edges():
    cu = int(G.nodes[u].get("cluster", -1))
    cv = int(G.nodes[v].get("cluster", -1))
    if cu in top_clusters and cv in top_clusters:
        key = (min(cu, cv), max(cu, cv))
        edge_counts[(cu, cv)] = edge_counts.get((cu, cv), 0) + 1
        if cu != cv:
            edge_counts[(cv, cu)] = edge_counts.get((cv, cu), 0) + 1

# Normalize by geometric mean of cluster sizes (density)
matrix = np.zeros((TOP_N, TOP_N))
for i, a in enumerate(top_clusters):
    for j, b in enumerate(top_clusters):
        raw = edge_counts.get((a, b), 0)
        if i == j:
            denom = size_map[a] * (size_map[a] - 1) / 2
        else:
            denom = size_map[a] * size_map[b]
        matrix[i][j] = raw / denom if denom > 0 else 0

# Short labels
topic_map = {}
with open(OUTPUT_DIR / "articles_leiden_cluster_report.csv") as f:
    for row in csv.DictReader(f):
        topic_map[int(row["cluster_id"])] = row["topic_label"]

labels = [f"C{c}\n{topic_map.get(c,'')[:22]}" for c in top_clusters]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matrix, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax, label="Edge density (edges / possible pairs)")
ax.set_xticks(range(TOP_N))
ax.set_yticks(range(TOP_N))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
ax.set_title("Inter-cluster edge density (top 12 clusters)\nDiagonal = intra-cluster density", fontsize=11)

for i in range(TOP_N):
    for j in range(TOP_N):
        val = matrix[i][j]
        if val > 0:
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=6, color="black" if val < matrix.max()*0.6 else "white")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_cluster_edge_density.png", dpi=150, bbox_inches="tight")

# Print isolation score: ratio of intra to total edges for each cluster
print("Cluster isolation (intra-cluster edges / all edges touching cluster):")
for i, c in enumerate(top_clusters):
    intra = edge_counts.get((c, c), 0)
    total_touching = sum(edge_counts.get((c, other), 0) for other in top_clusters) + intra
    if total_touching > 0:
        isolation = intra / total_touching
        print(f"  C{c:>3} ({topic_map.get(c,'')[:35]:<35}): {isolation:.2%} isolated")

Cluster isolation (intra-cluster edges / all edges touching cluster):
  C  0 (work stealing, load balancing, spee): 48.95% isolated
  C  1 (mixed strategy, backward induction,): 48.65% isolated
  C  2 (policy head, value head, residual n): 46.47% isolated
  C  3 (turing machine, problem space, prod): 46.72% isolated
  C  4 (discrimination net, expert memory, ): 48.61% isolated
  C  5 (chess instruction, critical thinkin): 49.11% isolated
  C  6 (counterfactual regret minimization,): 47.62% isolated
  C  7 (proof-number search, pn search, zob): 47.65% isolated
  C  8 (td learning, inductive logic progra): 48.86% isolated
  C  9 (piece-square tables, uci protocol, ): 48.78% isolated
  C 10 (null-move search, pv node, knowledg): 48.06% isolated
  C 11 (computer vision, image processing, ): 49.62% isolated


**INSIGHT 3 — Temporal profiles.** Bubble chart of each cluster's median year vs. its size, showing which research topics are old, new, large, or niche.

In [172]:
# INSIGHT 3 — Temporal profile: when did each research topic peak?
# Bubble chart: cluster median year (x) vs cluster size (bubble area),
# coloured by temporal spread (std). Reveals old-but-stable vs new-and-tight.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

year_df = pd.read_csv(OUTPUT_DIR / "articles_leiden_cluster_year_deviation.csv")
year_df = year_df[year_df["view"] == "keywords"].copy()
year_df = year_df[year_df["articles_with_year"] >= 3].dropna(subset=["median_year", "std_year"])

topic_map = {}
with open(OUTPUT_DIR / "articles_leiden_cluster_report.csv") as f:
    for row in csv.DictReader(f):
        topic_map[int(row["cluster_id"])] = row["topic_label"]

year_df["topic"] = year_df["cluster_id"].map(topic_map)

fig, ax = plt.subplots(figsize=(14, 7))

std_max = year_df["std_year"].max()
colors = cm.plasma_r(year_df["std_year"] / std_max)
sizes = (year_df["cluster_size"] / year_df["cluster_size"].max() * 1200 + 40).values

scatter = ax.scatter(
    year_df["median_year"], year_df["cluster_id"],
    s=sizes, c=year_df["std_year"], cmap="plasma_r",
    alpha=0.8, edgecolors="white", linewidths=0.4,
)
plt.colorbar(scatter, ax=ax, label="Temporal spread (std of years)")

# Label the large clusters
for _, row in year_df[year_df["cluster_size"] >= 20].iterrows():
    label = f"C{int(row['cluster_id'])}: {row['topic'][:28]}"
    ax.annotate(label, (row["median_year"], row["cluster_id"]),
                fontsize=7, ha="left", va="center",
                xytext=(6, 0), textcoords="offset points",
                color="#cccccc")

ax.set_xlabel("Median publication year", fontsize=11)
ax.set_ylabel("Cluster ID", fontsize=11)
ax.set_title("Temporal profile of research clusters\n(bubble size = cluster size, color = temporal spread)", fontsize=11)
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#16213e")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_cluster_temporal_profile.png", dpi=150, bbox_inches="tight")

# Print the most time-coherent vs most time-diverse large clusters
large = year_df[year_df["cluster_size"] >= 15].sort_values("std_year")
print("Most temporally COHERENT large clusters (tight publication window):")
for _, r in large.head(5).iterrows():
    print(f"  C{int(r.cluster_id):>3}  median={int(r.median_year)}  std={r.std_year:.1f}  size={int(r.cluster_size)}  {r.topic[:50]}")
print()
print("Most temporally DIVERSE large clusters (span many decades):")
for _, r in large.tail(5).iterrows():
    print(f"  C{int(r.cluster_id):>3}  median={int(r.median_year)}  std={r.std_year:.1f}  range={int(r.min_year)}-{int(r.max_year)}  {r.topic[:50]}")

Most temporally COHERENT large clusters (tight publication window):
  C 21  median=2024  std=2.0  size=20  large language model, language model, llm
  C 20  median=2024  std=4.7  size=24  transformer, transformer architecture, transformer
  C 22  median=2016  std=5.1  size=16  aesthetic evaluation, chess problem composition, a
  C 19  median=2007  std=5.3  size=24  fitness function, mutation, genetic programming
  C 17  median=2005  std=5.4  size=26  depth to mate, depth to conversion, dtz

Most temporally DIVERSE large clusters (span many decades):
  C  9  median=2009  std=13.4  range=1963-2024  piece-square tables, uci protocol, rybka
  C 10  median=1998  std=13.6  range=1977-2021  null-move search, pv node, knowledge-based system
  C  0  median=2000  std=13.9  range=1938-2020  work stealing, load balancing, speedup
  C  3  median=2006  std=13.9  range=1957-2023  turing machine, problem space, production system
  C 12  median=2002  std=14.2  range=1973-2024  chess chip, knowledge, fp

**INSIGHT 3 (alt) — Year boxplots.** Alternative view of temporal profiles as horizontal boxplots showing the full publication-year distribution (quartiles and outliers) per cluster.

In [173]:
# INSIGHT 3 (alt) — Temporal profile as horizontal boxplots
# One box per cluster sorted by median year.
# Shows median, IQR, whiskers, and outliers — no encoding tricks needed.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

def _to_year_i3(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        y = int(float(s))
        if 1500 <= y <= 2100:
            return y
    except Exception:
        return None
    return None

topic_map_i3 = {}
with open(OUTPUT_DIR / "articles_leiden_cluster_report.csv") as f:
    for row in csv.DictReader(f):
        topic_map_i3[int(row["cluster_id"])] = row["topic_label"]

article_year_i3 = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    y = _to_year_i3(row.get("crossref_year")) or _to_year_i3(row.get("original_year")) or _to_year(row.get("openalex_year"))
    article_year_i3[aid] = y

cluster_years_i3 = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid < 0:
        continue
    y = article_year_i3.get(n)
    if y:
        cluster_years_i3.setdefault(cid, []).append(y)

cluster_years_i3 = {cid: yrs for cid, yrs in cluster_years_i3.items() if len(yrs) >= 3}

sorted_clusters_i3 = sorted(cluster_years_i3.items(), key=lambda x: np.median(x[1]))

labels_i3, data_i3, sizes_i3 = [], [], []
for cid, yrs in sorted_clusters_i3:
    topic = topic_map_i3.get(cid, f"C{cid}")[:35]
    labels_i3.append(f"C{cid}: {topic}  (n={len(yrs)})")
    data_i3.append(yrs)
    sizes_i3.append(len(yrs))

max_size_i3 = max(sizes_i3)
cmap_i3 = plt.cm.YlOrRd
colors_i3 = [cmap_i3(s / max_size_i3 * 0.75 + 0.25) for s in sizes_i3]

fig, ax = plt.subplots(figsize=(13, max(7, len(data_i3) * 0.45)))

bp = ax.boxplot(
    data_i3, vert=False, patch_artist=True,
    medianprops=dict(color="white", linewidth=2),
    whiskerprops=dict(color="#aaaaaa", linewidth=1),
    capprops=dict(color="#aaaaaa", linewidth=1),
    flierprops=dict(marker="o", markerfacecolor="#ff6b6b", markersize=3, alpha=0.6, linestyle="none"),
)

for patch, color in zip(bp["boxes"], colors_i3):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

ax.set_yticks(range(1, len(labels_i3) + 1))
ax.set_yticklabels(labels_i3, fontsize=8)
ax.set_xlabel("Publication year", fontsize=11)
ax.set_title("Temporal profile of research clusters (sorted by median year · box = IQR · whiskers = 1.5×IQR · dots = outliers · color = cluster size)",
    fontsize=11,
)

ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#16213e")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")

sm = plt.cm.ScalarMappable(cmap=cmap_i3, norm=plt.Normalize(vmin=0, vmax=max_size_i3))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Cluster size", shrink=0.6)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_cluster_temporal_boxplot.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())


In [174]:
# INSIGHT 4 — Hub age: are the most-connected papers old or new?
# For each large cluster finds the highest-degree node and its publication year.
# Old hubs = foundational topic; young hubs = recently emerged topic.


def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        y = int(float(s))
        if 1500 <= y <= 2100:
            return y
    except Exception:
        return None
    return None

article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    y = _to_year(row.get("crossref_year")) or _to_year(row.get("original_year")) or _to_year(row.get("openalex_year"))
    article_year[aid] = y

topic_map = {}
with open(OUTPUT_DIR / "articles_leiden_cluster_report.csv") as f:
    for row in csv.DictReader(f):
        topic_map[int(row["cluster_id"])] = row["topic_label"]

size_map = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        size_map[cid] = size_map.get(cid, 0) + 1

rows = []
for cid, csize in sorted(size_map.items(), key=lambda x: -x[1]):
    if csize < 10:
        continue
    cluster_nodes = [n for n, d in G.nodes(data=True) if int(d.get("cluster", -1)) == cid]
    hub = max(cluster_nodes, key=lambda n: G.degree(n))
    hub_year = article_year.get(hub)
    hub_title = G.nodes[hub].get("title", hub)[:60]
    cluster_years = [article_year[n] for n in cluster_nodes if article_year.get(n)]
    med_year = sorted(cluster_years)[len(cluster_years)//2] if cluster_years else None
    rows.append({
        "cluster": f"C{cid}",
        "size": csize,
        "topic": topic_map.get(cid, "")[:40],
        "hub_degree": G.degree(hub),
        "hub_year": hub_year,
        "cluster_median_year": med_year,
        "hub_title": hub_title,
        "hub_older_than_median": (hub_year < med_year - 5) if hub_year and med_year else None,
    })

hub_df = pd.DataFrame(rows)
print(hub_df[["cluster","size","topic","hub_degree","hub_year","cluster_median_year","hub_older_than_median"]].to_string(index=False))

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(hub_df))
has_year = hub_df["hub_year"].notna() & hub_df["cluster_median_year"].notna()
df_plot = hub_df[has_year].copy()
y_pos = np.arange(len(df_plot))

bar_colors = ["#e74c3c" if row["hub_older_than_median"] else "#2ecc71"
              for _, row in df_plot.iterrows()]

ax.barh(y_pos, df_plot["hub_year"], color=bar_colors, alpha=0.8, label="Hub (most-connected paper) year")
ax.scatter(df_plot["cluster_median_year"], y_pos, color="white", zorder=5, s=40, label="Cluster median year")

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r.cluster} {r.topic[:30]}" for _, r in df_plot.iterrows()], fontsize=8)
ax.set_xlabel("Publication year")
ax.set_title("Hub age vs cluster median year\nRed bar = hub is older than cluster median (foundational anchor)", fontsize=10)
ax.legend(fontsize=9)

ax.legend(handles=[
    mpatches.Patch(color="#e74c3c", label="Hub older than cluster median (foundational paper)"),
    mpatches.Patch(color="#2ecc71", label="Hub younger / same era as cluster"),
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="white", markersize=6, label="Cluster median year"),
], fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_hub_age.png", dpi=150, bbox_inches="tight")


cluster  size                                    topic  hub_degree  hub_year  cluster_median_year hub_older_than_median
     C0    54   work stealing, load balancing, speedup           8    2004.0                 2000                 False
     C1    48 mixed strategy, backward induction, winn           8    2002.0                 2002                 False
     C2    48 policy head, value head, residual networ           8    2019.0                 2022                 False
     C3    47 turing machine, problem space, productio           8    2021.0                 2006                 False
     C4    46 discrimination net, expert memory, retri           8    2001.0                 2003                 False
     C5    46 chess instruction, critical thinking, cl           7    2013.0                 2017                 False
     C6    45 counterfactual regret minimization, reco           8    2006.0                 2020                  True
     C7    43 proof-number search, pn se

**INSIGHT 5 — Deep learning paradigm shift.** Side-by-side year histograms for the TD-learning and deep RL clusters pinpoint when the field transitioned from classical to neural approaches.

In [175]:
# INSIGHT 5 — The deep learning paradigm shift: TD-learning (C6) → Deep RL (C3)
# Side-by-side year histograms show when each community published.
# Cross-cluster edges between C3 and C6 reveal bridge papers at the transition.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        return int(float(s)) if 1500 <= int(float(s)) <= 2100 else None
    except Exception:
        return None

article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    article_year[aid] = _to_year(row.get("crossref_year")) or _to_year(row.get("original_year")) or _to_year(row.get("openalex_year"))

PARADIGM_CLUSTERS = {
    "C1: Parallel search": 1,
    "C3: Deep RL (AlphaZero)": 3,
    "C6: TD-learning / symbolic ML": 6,
    "C9: Classical alpha-beta": 9,
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=False)
fig.suptitle("Publication year distributions across AI paradigm clusters", fontsize=12, fontweight="bold")

for ax, (label, cid) in zip(axes.flat, PARADIGM_CLUSTERS.items()):
    years = [article_year[n] for n, d in G.nodes(data=True)
             if int(d.get("cluster", -1)) == cid and article_year.get(n)]
    if not years:
        continue
    ax.hist(years, bins=range(min(years), max(years)+2), color="#3498db", edgecolor="#1a252f", alpha=0.85)
    ax.axvline(sorted(years)[len(years)//2], color="#e74c3c", linestyle="--", linewidth=1.5,
               label=f"median={sorted(years)[len(years)//2]}")
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Articles")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_paradigm_timelines.png", dpi=150, bbox_inches="tight")

# Cross-cluster edges between C6 and C3 (bridge papers)
bridges = []
for u, v, ed in G.edges(data=True):
    cu = int(G.nodes[u].get("cluster", -1))
    cv = int(G.nodes[v].get("cluster", -1))
    if (cu == 3 and cv == 6) or (cu == 6 and cv == 3):
        for node in [u, v]:
            bridges.append({
                "node": node,
                "title": G.nodes[node].get("title", node)[:70],
                "year": article_year.get(node),
                "cluster": int(G.nodes[node].get("cluster", -1)),
                "degree": G.degree(node),
            })

bridge_df = pd.DataFrame(bridges).drop_duplicates("node")
if not bridge_df.empty:
    bridge_df = bridge_df.sort_values("year")
print(f"Cross-cluster edges between C3 (deep RL) and C6 (TD-learning): {len(G.edges())}")
print(f"Bridge papers (appear in C3↔C6 edges): {len(bridge_df)}")
if not bridge_df.empty:
    print(bridge_df[["title","year","cluster","degree"]].to_string(index=False))

Cross-cluster edges between C3 (deep RL) and C6 (TD-learning): 1819
Bridge papers (appear in C3↔C6 edges): 0


**INSIGHT 6 — Foundational anchors.** Finds old, high-degree papers embedded inside modern clusters — articles published decades ago that still sit at the structural core of active research.

In [ ]:
# INSIGHT 6 — Foundational anchors: old high-degree papers embedded in modern clusters
# For each large cluster, finds papers published ≥15 years before the cluster median
# that still have above-average degree. These are the "red hubs" in the year-colored graph.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        return int(float(s)) if 1500 <= int(float(s)) <= 2100 else None
    except Exception:
        return None

article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    article_year[aid] = _to_year(row.get("crossref_year")) or _to_year(row.get("original_year")) or _to_year(row.get("openalex_year"))

topic_map = {}
with open(OUTPUT_DIR / "articles_leiden_cluster_report.csv") as f:
    for row in csv.DictReader(f):
        topic_map[int(row["cluster_id"])] = row["topic_label"]

size_map = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        size_map[cid] = size_map.get(cid, 0) + 1

anchors = []
for cid, csize in sorted(size_map.items(), key=lambda x: -x[1]):
    if csize < 15:
        continue
    cluster_nodes = [n for n, d in G.nodes(data=True) if int(d.get("cluster", -1)) == cid]
    years = sorted([article_year[n] for n in cluster_nodes if article_year.get(n)])
    if not years:
        continue
    median_y = years[len(years)//2]
    avg_deg = sum(G.degree(n) for n in cluster_nodes) / len(cluster_nodes)
    for n in cluster_nodes:
        y = article_year.get(n)
        if y and y <= median_y - 15 and G.degree(n) >= avg_deg:
            anchors.append({
                "cluster": f"C{cid}",
                "topic": topic_map.get(cid, "")[:40],
                "cluster_median_year": median_y,
                "paper_year": y,
                "years_before_median": median_y - y,
                "degree": G.degree(n),
                "title": G.nodes[n].get("title", n)[:75],
            })

anchor_df = pd.DataFrame(anchors).sort_values("years_before_median", ascending=False)
print("Foundational anchors (published ≥15 yrs before cluster median, degree ≥ cluster avg):")
print(anchor_df[["cluster","topic","paper_year","cluster_median_year","years_before_median","degree","title"]].to_string(index=False))

if not anchor_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(anchor_df)*0.4)))
    colors = plt.cm.RdYlGn_r(anchor_df["years_before_median"] / anchor_df["years_before_median"].max())
    bars = ax.barh(range(len(anchor_df)), anchor_df["years_before_median"], color=colors, alpha=0.9)
    ax.set_yticks(range(len(anchor_df)))
    ax.set_yticklabels(
        [f"{r['cluster']} ({r['paper_year']}) {r['title'][:45]}" for _, r in anchor_df.iterrows()],
        fontsize=8
    )
    ax.set_xlabel("Years before cluster median (how much older than typical cluster paper)")
    ax.set_title("Foundational anchor papers\n(old, well-connected papers embedded in later research communities)", fontsize=10)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "insight_foundational_anchors.png", dpi=150, bbox_inches="tight")


Foundational anchors (published ≥15 yrs before cluster median, degree ≥ cluster avg):
cluster                                    topic  paper_year  cluster_median_year  years_before_median  degree                                                                         title
     C3 turing machine, problem space, productio        1957                 2006                   49       5                                                         STRAIGHT TO THE HEART
     C1 mixed strategy, backward induction, winn        1967                 2002                   35       6   Games with Incomplete Information Played by “Bayesian” Players, I–III Part 
     C3 turing machine, problem space, productio        1971                 2006                   35       3                        Some characteristics of human problem-solving in chess
     C4 discrimination net, expert memory, retri        1973                 2003                   30       5                                                

In [ ]:
# Foundational anchors — scatter visualisation (standalone cell)
# Loads G from saved GraphML and meta from CSV — no other cells required.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

def normalize_id_from_pdf(pdf_name):
    """Normalize article ID from a PDF filename."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    if s.lower().endswith(".pdf"):
        s = s[:-4]
    return s.lower()


def _to_year(value):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 4:
        y = int(digits[:4])
        if 1500 <= y <= 2100:
            return y
    try:
        return int(float(s)) if 1500 <= int(float(s)) <= 2100 else None
    except Exception:
        return None


# Load data
meta = pd.read_csv(META_PATH)
G = nx.read_graphml(GRAPHML_OUT)

article_year = {}
for _, row in meta.iterrows():
    aid = normalize_id_from_pdf(row.get("pdf_file"))
    article_year[aid] = _to_year(row.get("crossref_year")) or _to_year(row.get("original_year")) or _to_year(row.get("openalex_year"))

topic_map = {}
with open(CLUSTER_REPORT_PATH) as f:
    for row in csv.DictReader(f):
        topic_map[int(row["cluster_id"])] = row["topic_label"]

# Cluster sizes
size_map = {}
for n, d in G.nodes(data=True):
    cid = int(d.get("cluster", -1))
    if cid >= 0:
        size_map[cid] = size_map.get(cid, 0) + 1

# Build per-paper records for large clusters
records = []
for cid, csize in size_map.items():
    if csize < 15:
        continue
    cluster_nodes = [n for n, d in G.nodes(data=True) if int(d.get("cluster", -1)) == cid]
    years = sorted([article_year[n] for n in cluster_nodes if article_year.get(n)])
    if not years:
        continue
    median_y = years[len(years) // 2]
    avg_deg = sum(G.degree(n) for n in cluster_nodes) / len(cluster_nodes)
    for n in cluster_nodes:
        y = article_year.get(n)
        if y:
            records.append({
                "cluster": cid,
                "label": f"C{cid}",
                "paper_year": y,
                "cluster_median_year": median_y,
                "degree": G.degree(n),
                "title": G.nodes[n].get("title", n)[:55],
                "is_anchor": (y <= median_y - 15 and G.degree(n) >= avg_deg),
                "is_young_hub": (y >= median_y + 10 and G.degree(n) >= avg_deg * 2),
            })

df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(11, 9))

bg = df[~df["is_anchor"] & ~df["is_young_hub"]]
ax.scatter(
    bg["paper_year"], bg["cluster_median_year"],
    s=bg["degree"] * 6 + 10,
    c="#b0bec5", alpha=0.25, linewidths=0, zorder=1, label="all papers"
)

anc = df[df["is_anchor"]]
ax.scatter(
    anc["paper_year"], anc["cluster_median_year"],
    s=anc["degree"] * 18 + 30,
    c="#e74c3c", alpha=0.85, linewidths=0.6, edgecolors="#900",
    zorder=3, label=f"foundational anchors (n={len(anc)})"
)

yh = df[df["is_young_hub"]]
ax.scatter(
    yh["paper_year"], yh["cluster_median_year"],
    s=yh["degree"] * 18 + 30,
    c="#27ae60", alpha=0.85, linewidths=0.6, edgecolors="#145a32",
    zorder=3, label=f"young hubs (n={len(yh)})"
)

all_years = pd.concat([df["paper_year"], df["cluster_median_year"]])
lim_lo, lim_hi = all_years.min() - 3, all_years.max() + 3
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "--", color="#555", linewidth=1.1,
        alpha=0.6, zorder=2, label="paper year = cluster median")
ax.plot([lim_lo, lim_hi], [lim_lo + 15, lim_hi + 15], ":", color="#e74c3c",
        linewidth=1.0, alpha=0.5, zorder=2, label="−15 yr threshold (anchor zone)")

for _, row in anc.sort_values("degree", ascending=False).head(6).iterrows():
    ax.annotate(
        f"{row['label']} {row['paper_year']}\n{row['title']}",
        xy=(row["paper_year"], row["cluster_median_year"]),
        xytext=(row["paper_year"] + 2, row["cluster_median_year"] + 1),
        fontsize=6.5, color="#7b241c",
        arrowprops=dict(arrowstyle="-", color="#e74c3c", lw=0.7),
    )

for _, row in yh.sort_values("degree", ascending=False).head(4).iterrows():
    ax.annotate(
        f"{row['label']} {row['paper_year']}\n{row['title']}",
        xy=(row["paper_year"], row["cluster_median_year"]),
        xytext=(row["paper_year"] - 2, row["cluster_median_year"] - 2),
        fontsize=6.5, color="#1a5276", ha="right",
        arrowprops=dict(arrowstyle="-", color="#27ae60", lw=0.7),
    )

ax.fill_between([lim_lo, lim_hi], [lim_lo, lim_hi], [lim_lo - 60, lim_hi - 60],
                color="#27ae60", alpha=0.04, zorder=0)
ax.fill_between([lim_lo, lim_hi], [lim_lo, lim_hi], [lim_lo + 60, lim_hi + 60],
                color="#e74c3c", alpha=0.04, zorder=0)

ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel("Paper publication year", fontsize=11)
ax.set_ylabel("Cluster median year", fontsize=11)
ax.set_title(
    "Temporal displacement of papers within their clusters\n"
    "Above diagonal = paper older than cluster median (anchor zone)  |  "
    "Below diagonal = paper newer than median (young hub zone)",
    fontsize=10, fontweight="bold"
)
ax.legend(fontsize=8, loc="upper left")
ax.set_aspect("equal")
ax.grid(True, linestyle=":", alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "insight_temporal_displacement.png", dpi=150, bbox_inches="tight")
print(f"Foundational anchors: {len(anc)}, Young hubs: {len(yh)}")


Foundational anchors: 18, Young hubs: 8


Above the diagonal (Y > X): the cluster's median year is later than the paper year. The paper was published early, but ended up inside a community that mostly formed later. These are the red anchor dots — old papers that got absorbed into a modern cluster and stayed well-connected.

Below the diagonal (Y < X): the cluster's median year is earlier than the paper year. The paper was published late, but landed in a community whose bulk was written earlier. These are the green hub dots — recent papers that became the central reference in an older field (e.g. the CFR paper in C5, or the robotics paper in C11 that attracted a burst of recent citations).

In [ ]:
# INSIGHT — Author career arcs
# For the top-20 most prolific authors (connected articles only),
# plots each article as a dot: x = year, y = author, colour = keyword cluster.
# Shows whether researchers stay in one community or migrate across clusters.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})


def _norm(pdf):
    """Normalize a PDF filename to a lowercase ID."""
    if pd.isna(pdf): 
        return ""
    s = str(pdf).strip()
    return s[:-4].lower() if s.lower().endswith(".pdf") else s.lower()

def _to_year(v):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(v): 
        return None
    d = "".join(c for c in str(v) if c.isdigit())
    if len(d) >= 4:
        y = int(d[:4])
        if 1500 <= y <= 2100: 
            return y

meta = pd.read_csv(META_PATH)
meta["article_id"] = meta["pdf_file"].map(_norm)
meta["year"] = meta.apply(
    lambda r: _to_year(r.get("crossref_year")) or _to_year(r.get("original_year")) or _to_year(r.get("openalex_year")),
    axis=1)

cr  = pd.read_csv(CR_PATH)
Gg  = nx.read_graphml(GRAPHML_OUT)
node_cid = {n: int(d.get("cluster", -1)) for n, d in Gg.nodes(data=True)}

# Expand one row per (author, article)
rows = []
for _, row in meta.iterrows():
    aid = row["article_id"]
    cid = node_cid.get(aid, -1)
    if cid < 0:
        continue  # skip isolated articles
    raw = row.get("crossref_authors") or row.get("original_authors") or ""
    if pd.isna(raw):
        continue
    for author in [a.strip() for a in str(raw).split(";") if a.strip() and not all(p.lower() == "nan" for p in a.strip().split())]:
        rows.append({"author": author, "article_id": aid,
                     "year": row["year"], "cluster_id": cid})

arc_df = pd.DataFrame(rows)
arc_df.to_csv(ARC_OUT, index=False)

# Top-20 authors by unique connected articles
top_authors = (arc_df.groupby("author")["article_id"]
               .nunique().nlargest(20).index.tolist())

plot_df = (arc_df[arc_df["author"].isin(top_authors) & arc_df["year"].notna()]
           .copy())
plot_df["year"] = plot_df["year"].astype(int)

all_cids = sorted(plot_df["cluster_id"].unique())
cmap     = cm.get_cmap("tab20", len(all_cids))
cid_col  = {c: cmap(i) for i, c in enumerate(all_cids)}

# Sort authors by their first publication year
author_order = (plot_df.groupby("author")["year"].min()
                .reindex(top_authors).sort_values().index.tolist())

fig, ax = plt.subplots(figsize=(15, 9))

for yi, author in enumerate(author_order):
    sub = plot_df[plot_df["author"] == author]
    for _, r in sub.iterrows():
        ax.scatter(r["year"], yi,
                   color=cid_col[r["cluster_id"]],
                   s=70, alpha=0.85, zorder=3,
                   edgecolors="white", linewidths=0.4)
    # thin line spanning the author's career
    yr_span = sub["year"].agg(["min", "max"])
    ax.plot([yr_span["min"], yr_span["max"]], [yi, yi],
            color="lightgray", linewidth=0.8, zorder=1)

ax.set_yticks(range(len(author_order)))
ax.set_yticklabels(author_order, fontsize=9)
ax.set_xlabel("Publication year", fontsize=11)
ax.set_title(
    "Author Career Arcs — top 20 most prolific authors\n"
    "Each dot = one article in the keyword graph   |   Colour = keyword cluster",
    fontweight="bold", fontsize=12)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

# Legend: top-12 clusters only to avoid clutter
cr_map = {int(r["cluster_id"]): r["topic_label"].split(",")[0].strip()[:30]
          for _, r in cr.iterrows()}
top12  = cr.nlargest(12, "cluster_size")["cluster_id"].astype(int).tolist()
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=cid_col.get(c, "gray"), markersize=9,
               label=f"C{c}: {cr_map.get(c, '?')}")
    for c in top12 if c in cid_col
]
ax.legend(handles=handles, loc="upper left", fontsize=7.5,
          title="Top-12 clusters", title_fontsize=8.5,
          framealpha=0.88, ncol=2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "author_career_arcs.png",
            dpi=150, bbox_inches="tight")
print(f"Author career arcs saved. {len(top_authors)} authors plotted.")


/tmp/ipykernel_5562/580815769.py:65: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap     = cm.get_cmap("tab20", len(all_cids))


Author career arcs saved. 20 authors plotted.


**Keyword specificity spectrum.** Ranks every keyword by concentration: niche terms appear in a single cluster while broad concepts are spread across many, distinguishing specialist from cross-cutting vocabulary.

In [ ]:
# INSIGHT — Keyword specificity spectrum
# For every keyword: total article count + cluster spread (distinct clusters).
# Scatter: cluster spread (x) vs frequency (y, log scale).
# Top-right  = bridge vocabulary  (frequent + wide spread across communities)
# Bottom-left = niche jargon      (rare + confined to a single cluster)

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})


def _parse_kw(cell):
    """Parse keyword-strength pairs from a raw cell string."""
    if pd.isna(cell): 
        return []
    try:
        p = ast.literal_eval(str(cell).strip())
        return [str(x[0]).strip().lower() for x in p
                if isinstance(x, (list, tuple)) and len(x) >= 2]
    except Exception:
        return []

kw_df = pd.read_csv(KW_PATH)
kw_df["article_id"] = kw_df["filename"].map(lambda x: str(x).strip().lower())
kw_df["kw_list"]    = kw_df["keywords_without_stopwords"].map(_parse_kw)

Gg = nx.read_graphml(GRAPHML_OUT)
node_cid = {n: int(d.get("cluster", -1)) for n, d in Gg.nodes(data=True)}

kw_articles = defaultdict(set)
kw_clusters = defaultdict(set)

for _, row in kw_df.iterrows():
    aid = row["article_id"]
    cid = node_cid.get(aid, -1)
    for kw in row["kw_list"]:
        kw_articles[kw].add(aid)
        if cid >= 0:
            kw_clusters[kw].add(cid)

spec_df = pd.DataFrame([
    {"keyword":        kw,
     "total_articles": len(arts),
     "cluster_spread": len(kw_clusters.get(kw, set()))}
    for kw, arts in kw_articles.items()
]).sort_values("total_articles", ascending=False)

spec_df.to_csv(SPEC_OUT, index=False)
print(f"Total unique keywords: {len(spec_df)}")
print(f"Spread = 1 (single-cluster jargon): {(spec_df.cluster_spread==1).sum()}")
print(f"Spread >= 10 (bridge terms):         {(spec_df.cluster_spread>=10).sum()}")
print("Saved:", SPEC_OUT)

# ── Plot ──────────────────────────────────────────────────────────────────
plot_df = spec_df[spec_df["total_articles"] >= 3].copy()

fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(
    plot_df["cluster_spread"],
    plot_df["total_articles"],
    c=plot_df["cluster_spread"],
    cmap="plasma",
    alpha=0.35, s=20,
    edgecolors="none",
)
plt.colorbar(sc, ax=ax, label="Cluster spread (distinct clusters)")
ax.set_yscale("log")
ax.set_xlabel("Cluster spread  (number of distinct clusters the keyword appears in)",
              fontsize=11)
ax.set_ylabel("Total articles using this keyword  (log scale)", fontsize=11)
ax.set_title(
    "Keyword Specificity Spectrum\n"
    "Top-right = bridge vocabulary (frequent + wide)   "
    "Bottom-left = niche jargon (rare + single-cluster)",
    fontweight="bold", fontsize=12)

# ── annotate bridge terms (top-right quadrant) ────────────────────────────
bridge_df = plot_df[
    (plot_df["cluster_spread"] >= plot_df["cluster_spread"].quantile(0.96)) &
    (plot_df["total_articles"] >= plot_df["total_articles"].quantile(0.80))
].nlargest(20, "total_articles")

for _, row in bridge_df.iterrows():
    ax.annotate(
        row["keyword"],
        xy=(row["cluster_spread"], row["total_articles"]),
        xytext=(5, 2), textcoords="offset points",
        fontsize=8, color="#c0392b", fontweight="bold",
    )

# ── annotate high-frequency single-cluster jargon ─────────────────────────
jargon_df = plot_df[
    (plot_df["cluster_spread"] <= 2) &
    (plot_df["total_articles"] >= plot_df["total_articles"].quantile(0.94))
].nlargest(12, "total_articles")

for _, row in jargon_df.iterrows():
    ax.annotate(
        row["keyword"],
        xy=(row["cluster_spread"], row["total_articles"]),
        xytext=(5, 2), textcoords="offset points",
        fontsize=8, color="#2471a3",
    )

# ── reference lines ───────────────────────────────────────────────────────
ax.axvline(1,  color="#2471a3", linewidth=0.8, linestyle="--", alpha=0.5,
           label="Spread = 1  (pure jargon)")
ax.axvline(10, color="#c0392b", linewidth=0.8, linestyle="--", alpha=0.5,
           label="Spread ≥ 10  (bridge term)")
ax.legend(fontsize=9, loc="upper left", framealpha=0.85)
ax.grid(alpha=0.18)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "keyword_specificity_spectrum.png",
            dpi=150, bbox_inches="tight")

# ── summary table ─────────────────────────────────────────────────────────
print("\nTop-15 bridge terms (highest cluster spread, ≥5 articles):")
print(spec_df[spec_df["total_articles"] >= 5]
      .nlargest(15, "cluster_spread")
      [["keyword","total_articles","cluster_spread"]].to_string(index=False))

print("\nTop-15 high-frequency jargon (spread ≤ 2, ≥5 articles):")
print(spec_df[(spec_df["cluster_spread"] <= 2) & (spec_df["total_articles"] >= 5)]
      .nlargest(15, "total_articles")
      [["keyword","total_articles","cluster_spread"]].to_string(index=False))


Total unique keywords: 33856
Spread = 1 (single-cluster jargon): 17504
Spread >= 10 (bridge terms):         53
Saved: /home/john/repos/leno4ka/output/keyword_specificity_spectrum.csv

Top-15 bridge terms (highest cluster spread, ≥5 articles):
                 keyword  total_articles  cluster_spread
                checkers              32              16
              middlegame              32              15
          search process              33              14
            move quality              32              14
computational complexity              33              13
                 opening              32              13
     cognitive processes              31              13
        pattern matching              28              13
                symmetry              26              13
              backgammon              32              12
       search efficiency              29              12
     confidence interval              22              12
          expect

**Sankey diagram.** Shows how articles flow between keyword, author, and combined cluster assignments, making cross-view agreement or divergence visually explicit.

In [ ]:
# INSIGHT — Sankey: keyword → author → combined cluster mapping
# Each article has three cluster labels (keyword view, author view, combined view).
# Ribbons show how keyword communities split or merge across the other two views.
# Saved as interactive HTML via Plotly.

def _get_cid(d):
    """Extract integer cluster ID from a node attribute dict."""
    v = d.get("cluster", -1)
    try: return int(float(v))
    except (TypeError, ValueError):
         return -1

Gk = nx.read_graphml(KW_CLUSTER_GRAPHML_OUT)
Ga = nx.read_graphml(AUTH_GRAPHML_OUT)
Gc = nx.read_graphml(COMB_GRAPHML_OUT)
kw_cr = pd.read_csv(CLUSTER_REPORT_OUT)
auth_cr = pd.read_csv(AUTH_REPORT_OUT)
comb_cr = pd.read_csv(COMB_REPORT_OUT)

TOP_KW, TOP_AU, TOP_CO = 10, 12, 8

top_kw = kw_cr.nlargest(TOP_KW,   "cluster_size")["cluster_id"].astype(int).tolist()
top_au = auth_cr.nlargest(TOP_AU, "cluster_size")["cluster_id"].astype(int).tolist()
top_co = comb_cr.nlargest(TOP_CO, "cluster_size")["cluster_id"].astype(int).tolist()

node_kw = {n: _get_cid(d) for n, d in Gk.nodes(data=True)}
node_au = {n: _get_cid(d) for n, d in Ga.nodes(data=True)}
node_co = {n: _get_cid(d) for n, d in Gc.nodes(data=True)}

kw_lbl   = {int(r["cluster_id"]): r["topic_label"].split(",")[0].strip()[:22]
            for _, r in kw_cr.iterrows()}

kw_idx = {c: i               for i, c in enumerate(top_kw)}
au_idx = {c: TOP_KW + i      for i, c in enumerate(top_au)}
co_idx = {c: TOP_KW+TOP_AU+i for i, c in enumerate(top_co)}

flows_ka, flows_ac = {}, {}
for n in Gk.nodes():
    kc = node_kw.get(n, -1)
    ac = node_au.get(n, -1)
    cc = node_co.get(n, -1)
    if kc in kw_idx and ac in au_idx:
        flows_ka[(kc, ac)] = flows_ka.get((kc, ac), 0) + 1
    if ac in au_idx and cc in co_idx:
        flows_ac[(ac, cc)] = flows_ac.get((ac, cc), 0) + 1

src, tgt, val, link_colors = [], [], [], []
for (kc, ac), cnt in flows_ka.items():
    if cnt < 2: continue
    src.append(kw_idx[kc]); tgt.append(au_idx[ac]); val.append(cnt)
    link_colors.append("rgba(220,80,80,0.25)")
for (ac, cc), cnt in flows_ac.items():
    if cnt < 2: continue
    src.append(au_idx[ac]); tgt.append(co_idx[cc]); val.append(cnt)
    link_colors.append("rgba(60,120,200,0.25)")

def _hex(h, s=0.62, l=0.55):
    """Convert HSL components to a hex color string."""
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

node_labels = (
    [f"KW-C{c}: {kw_lbl.get(c,'?')}" for c in top_kw] +
    [f"Auth-C{c}"                      for c in top_au] +
    [f"Comb-C{c}"                      for c in top_co]
)
node_colors = (
    [_hex(i / TOP_KW)                 for i in range(TOP_KW)] +
    [_hex(0.38 + i / TOP_AU * 0.22)   for i in range(TOP_AU)] +
    [_hex(0.62 + i / TOP_CO * 0.18)   for i in range(TOP_CO)]
)

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=node_labels, color=node_colors, pad=16, thickness=20),
    link=dict(source=src, target=tgt, value=val, color=link_colors),
))
fig.update_layout(
    paper_bgcolor="white", plot_bgcolor="white",
    title_text=(
        "Sankey: Keyword Clusters \u2192 Author Clusters \u2192 Combined Clusters<br>"
        "<sub>Red ribbons = KW\u2192Auth  |  Blue = Auth\u2192Combined  |  Width = article count</sub>"
    ),
    font_size=11, height=750,
)
fig.write_image(SANKEY_PNG, width=1600, height=800, scale=2)
fig.write_html(SANKEY_OUT)
print("Sankey PNG saved:", SANKEY_PNG)
print("Sankey saved:", SANKEY_OUT)


Sankey PNG saved: /home/john/repos/leno4ka/output/sankey_kw_auth_comb.png
Sankey saved: /home/john/repos/leno4ka/output/sankey_kw_auth_comb.html


**Ego networks for anchor papers.** Draws the immediate neighbourhood of two foundational anchor papers to show which articles co-occur with them most closely in the graph.

In [ ]:
# INSIGHT — Ego network for foundational anchor papers
# Targets: 1957 paper in C0 (expert memory) and Harsanyi 1967 in C2 (game theory).
# Direct neighbours coloured red (old) → green (recent) by publication year.

def _norm(pdf):
    """Normalize a PDF filename to a lowercase ID."""
    if pd.isna(pdf): 
        return ""
    s = str(pdf).strip()
    return s[:-4].lower() if s.lower().endswith(".pdf") else s.lower()

def _to_year(v):
    """Parse a year value from various string formats, returning an int or None."""
    if pd.isna(v):
        return None
    d = "".join(c for c in str(v) if c.isdigit())
    if len(d) >= 4:
        y = int(d[:4])
        if 1500 <= y <= 2100:
            return y

def _get_cid(d):
    """Extract integer cluster ID from a node attribute dict."""
    v = d.get("cluster", -1)
    try: return int(float(v))
    except (TypeError, ValueError):
         return -1

def _year_hex(year, y_min, y_max):
    """Map a publication year to a red-green gradient hex color."""
    if year is None:
        return "#aaaaaa"
    t = max(0.0, min(1.0, (year - y_min) / max(y_max - y_min, 1)))
    r, g, b = colorsys.hls_to_rgb(t * 0.33, 0.50, 0.85)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

meta = pd.read_csv(META_PATH)
meta["article_id"] = meta["pdf_file"].map(_norm)
meta["year"] = meta.apply(
    lambda r: _to_year(r.get("crossref_year")) or _to_year(r.get("original_year")) or _to_year(r.get("openalex_year")),
    axis=1)
year_map = dict(zip(meta["article_id"], meta["year"]))

Gg = nx.read_graphml(GRAPHML_OUT)
node_cid = {n: _get_cid(d) for n, d in Gg.nodes(data=True)}

targets = {}
for n, d in Gg.nodes(data=True):
    cid = node_cid.get(n, -1)
    y   = year_map.get(n)
    if cid == 0 and y == 1957 and "C0_1957" not in targets:
        targets["C0_1957"] = n
    if cid == 2 and y == 1967 and "C2_1967" not in targets:
        targets["C2_1967"] = n

for label, cid, target_year in [("C0_1957", 0, 1957), ("C2_1967", 2, 1967)]:
    if label not in targets:
        cands = [(n, year_map.get(n)) for n in Gg.nodes()
                 if node_cid.get(n) == cid
                 and Gg.degree(n) > 0
                 and year_map.get(n) is not None
                 and not pd.isna(year_map.get(n))]
        if cands:
            best = min(cands, key=lambda x: abs(x[1] - target_year))
            targets[label] = best[0]
            print(f"  Fallback {label}: {Gg.nodes[best[0]].get('title','?')!r}  year={best[1]}")

print("Anchor papers:")
for lbl, n in targets.items():
    print(f"  {lbl}: {Gg.nodes[n].get('title','?')!r}  year={year_map.get(n)}")

all_years = [y for y in year_map.values() if y is not None and not pd.isna(y)]
y_min, y_max = min(all_years), max(all_years)

PYVIS_OPTS = (
    '{"physics":{"enabled":true,'
    '"barnesHut":{"gravitationalConstant":-8000,'
    '"centralGravity":0.3,"springLength":130},'
    '"stabilization":{"iterations":600}},'
    '"interaction":{"hover":true,"tooltipDelay":100}}'
)

for label, center in targets.items():
    neighbors = list(Gg.neighbors(center))
    subG = Gg.subgraph([center] + neighbors).copy()
    fname = OUTPUT_DIR / f"ego_{label}.html"

    net = Network(height="680px", width="100%",
                  bgcolor="white", font_color="black", notebook=False)
    net.set_options(PYVIS_OPTS)

    for n, nd in subG.nodes(data=True):
        y      = year_map.get(n)
        col    = _year_hex(y, y_min, y_max)
        is_ctr = (n == center)
        deg    = subG.degree(n)
        net.add_node(
            n,
            label=nd.get("title", n)[:28],
            title=(f"<b>{nd.get('title', n)}</b><br>"
                   f"Year: {y or '?'}<br>"
                   f"Cluster: C{node_cid.get(n, '?')}<br>"
                   f"Connections: {deg}"),
            color={"border": "#ffffff" if is_ctr else col, "background": col},
            size=28 if is_ctr else max(10, 8 + deg * 2),
            borderWidth=4 if is_ctr else 1,
            font={"size": 13 if is_ctr else 10},
        )
    for u, v, ed in subG.edges(data=True):
        s = float(ed.get("strength", 1.0))
        net.add_edge(u, v, width=max(0.6, s * 0.45),
                     color={"color": "#666666", "opacity": 0.65})

    net.write_html(fname, open_browser=False)
    cy = year_map.get(center)

    nbr_years = [y for nb in neighbors
                 if (y := year_map.get(nb)) is not None and not pd.isna(y)]
    med_nbr = int(np.median(nbr_years)) if nbr_years else "?"
    print(f"\n{label}  paper_year={cy}  median_neighbour_year={med_nbr}")
    print(f"  {subG.number_of_nodes()} nodes, {subG.number_of_edges()} edges -> {fname}")


  Fallback C0_1957: 'BOOK NOTICES'  year=1938.0
  Fallback C2_1967: 'Explanation-Based Learning and Reinforcement Learning: A Unified View'  year=1995.0
Anchor papers:
  C0_1957: 'BOOK NOTICES'  year=1938.0
  C2_1967: 'Explanation-Based Learning and Reinforcement Learning: A Unified View'  year=1995.0

C0_1957  paper_year=1938.0  median_neighbour_year=2015
  3 nodes, 3 edges -> /home/john/repos/leno4ka/output/ego_C0_1957.html

C2_1967  paper_year=1995.0  median_neighbour_year=2007
  6 nodes, 6 edges -> /home/john/repos/leno4ka/output/ego_C2_1967.html


**Venue concentration per cluster.** Stacked bar chart showing which journals dominate each keyword cluster, revealing whether clusters correspond to distinct publication venues.

In [ ]:
# INSIGHT — Venue / journal concentration per cluster
# Stacked bar for top-14 keyword clusters showing the 5 most common journals
# + everything else as "Other". Confirms whether clusters are also
# journal communities and strengthens the separate-citation-networks claim.

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})


def _norm(pdf):
    """Normalize a PDF filename to a lowercase ID."""
    if pd.isna(pdf): return ""
    s = str(pdf).strip()
    return s[:-4].lower() if s.lower().endswith(".pdf") else s.lower()

def _get_cid(d):
    """Extract integer cluster ID from a node attribute dict."""
    v = d.get("cluster", -1)
    try: return int(float(v))
    except (TypeError, ValueError): return -1

meta = pd.read_csv(META_PATH)
meta["article_id"] = meta["pdf_file"].map(_norm)
cr   = pd.read_csv(CR_PATH)
Gg   = nx.read_graphml(GRAPHML_OUT)

cid_map = {n: _get_cid(d) for n, d in Gg.nodes(data=True)}
meta["cluster_id"] = meta["article_id"].map(cid_map).fillna(-1).astype(int)

TOP_N    = 14
top_ids  = cr.nlargest(TOP_N, "cluster_size")["cluster_id"].astype(int).tolist()
cr_label = {int(r["cluster_id"]): r["topic_label"].split(",")[0].strip()[:20]
            for _, r in cr.iterrows()}

meta_top = meta[meta["cluster_id"].isin(top_ids)].copy()
meta_top["venue"] = meta_top["crossref_journal"].fillna("Unknown / Conference")

cl_venue = defaultdict(Counter)
for _, row in meta_top.iterrows():
    cl_venue[row["cluster_id"]][row["venue"]] += 1

global_cnt = Counter()
for c in cl_venue.values():
    global_cnt.update(c)
top5 = [v for v, _ in global_cnt.most_common(10)
        if v != "Unknown / Conference"][:5]

PALETTE = ["#e05b5b", "#f4a261", "#2a9d8f", "#457b9d", "#9b59b6", "#d0d0d0"]
vcol = {v: PALETTE[i] for i, v in enumerate(top5)}
vcol["Other"] = PALETTE[-1]

rows = []
for cid in top_ids:
    for venue, cnt in cl_venue[cid].most_common():
        rows.append({"cluster_id": cid, "topic": cr_label.get(cid, str(cid)),
                     "venue": venue, "count": cnt})
pd.DataFrame(rows).to_csv(VENUE_OUT, index=False)
print("Saved:", VENUE_OUT)

fig, ax = plt.subplots(figsize=(16, 7))
x       = np.arange(len(top_ids))
bottoms = np.zeros(len(top_ids))

for venue in top5 + ["Other"]:
    vals = []
    for cid in top_ids:
        cm = cl_venue[cid]
        if venue == "Other":
            vals.append(sum(v for k, v in cm.items() if k not in top5))
        else:
            vals.append(cm.get(venue, 0))
    lbl = venue[:42] if venue != "Other" else "Other / Unknown"
    ax.bar(x, vals, bottom=bottoms, width=0.65,
           color=vcol[venue], label=lbl, edgecolor="white", linewidth=0.4)
    bottoms += np.array(vals, dtype=float)

for i, cid in enumerate(top_ids):
    total = sum(cl_venue[cid].values())
    if total == 0:
        continue
    top_pct = cl_venue[cid].most_common(1)[0][1] / total * 100
    ax.text(i, bottoms[i] + 0.5, f"{top_pct:.0f}%",
            ha="center", va="bottom", fontsize=7.5, color="#333333")

ax.set_xticks(x)
ax.set_xticklabels(
    [f"C{c}\n{cr_label.get(c,'?')}" for c in top_ids],
    rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Article count")
ax.set_title(
    "Venue / Journal Concentration — top 14 keyword clusters\n"
    "% = share of the single most common venue per cluster",
    fontweight="bold", fontsize=12)
ax.legend(loc="upper right", fontsize=8, title="Venue / journal",
          title_fontsize=8.5, framealpha=0.88)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "venue_concentration_per_cluster.png",
            dpi=150, bbox_inches="tight")
